<center>МИНИСТЕРСТВО НАУКИ И ВЫСШЕГО ОБРАЗОВАНИЯ РОССИЙСКОЙ ФЕДЕРАЦИИ </center>
<center>ФЕДЕРАЛЬНОЕ ГОСУДАРСТВЕННОЕ БЮДЖЕТНОЕ ОБРАЗОВАТЕЛЬНОЕ УЧРЕЖДЕНИЕ ВЫСШЕГО ОБРАЗОВАНИЯ </center>
<center>«НОВОСИБИРСКИЙ ГОСУДАРСТВЕННЫЙ ТЕХНИЧЕСКИЙ УНИВЕРСИТЕТ»</center>
<center>Кафедра Вычислительной техники </center>
<br>
<center> <b> <font size="5">  ОТЧЁТ </font>  </b>  </center>   
<center><font size="3">по расчетно графической работе №1</font></center>
<center><font size="3">по дисциплине: «Интеллектуальные системы» </font></center>
<br>

Выполнил: Левицкий А.В., АПИМ2-26

Защищено: 03.11.26

Вариант: California Housing Data (1990)

<center>  Новосибирск, 2026  </center>


# РГР 1

Применение LLM для анализа данных и построения регрессионных моделей

## Тема работы

Применение больших языковых моделей (LLM) для предварительного анализа данных и построения регрессионных моделей с учётом характеристик исходного набора данных.

## Цель работы

Освоение приёмов составления запросов (промптов) к LLM для решения задач анализа данных и регрессии, оценка качества и корректности ответов LLM.

## Датасет California Houses

**Предметная область** - доход и размеры домашних хозяйств Калифорнии по данным до 1990г.
Признаки в таблице:
1) Median House Value: Медианная цена дома в квартале[$]
2) Median Income: Медианный доход на семью в квартале[10тыс.$] (TARGET)
3) Median Age: Медианный возраст дома в квартале; меньше = новее [лет]
4) Total Rooms: Общее количество комнат в квартале
5) Total Bedrooms: Общее количество спален в квартале
6) Population: Общее количество людей, проживающих в квартале
7) Households: Общее количество домохозяйств, групп людей, проживающих в жилом помещении, в квартале
8) Latitude: показатель того, насколько далеко на север находится дом; чем выше значение, тем дальше на север [°]
9) Longitude: показатель того, насколько далеко на запад находится дом; чем выше значение, тем дальше на запад [°]
10) Distance to coast: расстояние до ближайшей точки побережья [м]
11) Distance to Los Angeles: расстояние до центра Лос-Анджелеса [м]
12) Distance to San Diego: расстояние до центра Сан-Диего [м]
13) Distance to San Jose: расстояние до центра Сан-Хосе [м]
14) Distance to San Francisco: расстояние до центра Сан-Франциско [м]
15) Distance to the ocean (Расстояние до океана)

## Использованные LLM

- GPT-5.6 Luna
- DeepSeek-V4.1-Flash-1017

## Основная часть
---

### Формат работы

Структура каждого шага:
1. Промпт – текстовая (Markdown) ячейка с точным текстом запроса.
2. Ответ LLM – исполняемый фрагмент кода и/или текстовые пояснения.
3. Результат выполнения кода – вывод ячейки, таблицы, графики.
4. Комментарий бригады – оценка ответа: верен ли он, необходимые исправления, сколько потребовалось уточнений.


Сводка метрик:

In [ ]:
rows = 3
cols = 3
models_r2_data = [[0 for _ in range(cols)] for _ in range(rows)]

## GPT-5.6 Luna

### Анализ и подготовка датасета

**ПРОМПТ**:

```
Проведи предварительный анализ приложенного датасета. Определи:
- Символ, используемый для разделения ячеек
- Символ, используемый разделения дробной и целой части
- Наличие нечисловых данных
- Наличие штампов даты или времени
- Наличие заголовка

Создай Dataframe `df_gpt`, получи размеры исходной таблицы. Определи наличие пропусков и выбери способ их обработки. Получи размер таблицы после обработки пропусков.
Определи наличие и приведи нужные для работы признаки к числовому типу (цель работы - построение регрессионных моделей).
Сохрани в виде файла `dataset_prepared_gpt.csv` текущее состояние обработанной таблицы.

Работа ведется в Jupyter notebook - ответы представляй блоками по смыслу.

Библиотеки в дальнейшем: pandas, numpy, matplotlib, scikit-learn, statsmodels, TensorFlow/Keras
```

* Приложены оригинальные csv и пояснение к нему
* Включен режим *Think*

**Ответ GPT**

1. Результаты предварительного анализа

|Проверка|Результат|
|-|-|
|Разделитель столбцов|; — точка с запятой|
|Десятичный разделитель|	. — точка|
|Заголовок|	Присутствует, первая строка|
|Нечисловые данные|	Есть|
|Дата и время|	Не обнаружены|
|Исходный размер	20 640 строк × 16 столбцов|

Нечисловые данные:
- ocean_proximity — категориальный признак с пятью категориями.
- Median_Income — содержит некорректное значение ?$?.
- Также имеются пустые ячейки в Median_Income и Tot_Rooms.

Обратите внимание: фактический CSV содержит 16 столбцов, хотя приложенное описание перечисляет 15 признаков.

2. Создание DataFrame и загрузка данных

In [ ]:
import numpy as np
import pandas as pd

df_gpt = pd.read_csv(
    'California_Houses_15_col_dot.csv',
    sep=';',
    decimal='.',
    header=0,
    encoding='utf-8-sig'
)

print('Размер исходной таблицы:', df_gpt.shape)
display(df_gpt.head())

3. Проверка типов данных и пропусков

Параметр errors='coerce' преобразует некорректные числовые значения в NaN.

In [ ]:
# ocean_proximity — категориальный признак,
# остальные столбцы должны содержать числовые данные.

numeric_candidates = [
    col for col in df_gpt.columns
    if col != 'ocean_proximity'
]

for col in numeric_candidates:
    df_gpt[col] = pd.to_numeric(
        df_gpt[col],
        errors='coerce'
    )

print(df_gpt.dtypes)

In [ ]:
missing_before = df_gpt.isna().sum()

print('Количество пропусков по столбцам:')
print(missing_before[missing_before > 0])

print('Всего пропусков:',
      df_gpt.isna().sum().sum())

4. Обработка пропусков

В приложенном описании Median_Income обозначен как целевой признак (TARGET).

Поэтому выбрана следующая стратегия:
- Две строки с отсутствующим значением целевой переменной Median_Income удаляются. Заполнять целевую переменную медианой нежелательно, поскольку это создаёт искусственные значения для регрессии.
- Пропуск в Tot_Rooms заполняется медианой этого признака — 2127.

In [ ]:
target_col = 'Median_Income'

# Удаляем строки с отсутствующей целевой переменной
df_gpt = df_gpt.dropna(
    subset=[target_col]
).copy()

# Выбираем числовые признаки, исключая целевую переменную
feature_numeric_cols = [
    col for col in df_gpt.select_dtypes(
        include=[np.number]
    ).columns
    if col != target_col
]

# Заполняем пропуски медианами
df_gpt[feature_numeric_cols] = (
    df_gpt[feature_numeric_cols].fillna(
        df_gpt[feature_numeric_cols].median()
    )
)

print('Пропусков после обработки:',
      df_gpt.isna().sum().sum())

print('Размер таблицы после обработки:',
      df_gpt.shape)

5. Подготовка признаков для регрессионных моделей

При проверке обнаружено, что Median_House_Value и median_house_value содержат абсолютно одинаковые значения. Оставляем один столбец — Median_House_Value.

Категориальный признак ocean_proximity преобразуем с помощью One-Hot Encoding. Это позволит использовать категориальную информацию в регрессионных моделях.

In [ ]:
# Проверяем совпадение столбцов
print(
    df_gpt['Median_House_Value'].equals(
        df_gpt['median_house_value']
    )
)

# Удаляем полностью дублирующий столбец
df_gpt = df_gpt.drop(
    columns=['median_house_value']
)

# Кодируем категориальный признак
df_gpt = pd.get_dummies(
    df_gpt,
    columns=['ocean_proximity'],
    drop_first=True,
    dtype='int8'
)

print('Размер итоговой таблицы:', df_gpt.shape)
print('Количество нечисловых столбцов:',
      len(df_gpt.select_dtypes(exclude=[np.number]).columns))

display(df_gpt.head())

6. Сохранение подготовленного датасета

In [ ]:
df_gpt.to_csv(
    'dataset_prepared_gpt.csv',
    sep=';',
    decimal='.',
    index=False,
    encoding='utf-8-sig'
)

print('Файл dataset_prepared_gpt.csv сохранён')
print('Размер:', df_gpt.shape)

7. Итоговые результаты

Исходная таблица
20 640 × 16

После обработки пропусков
20 638 × 16

Итоговая таблица
20 638 × 18

Оставшиеся пропуски
0

Состояние итогового DataFrame df_gpt:
- Все 18 столбцов имеют числовой тип.
- Целевой признак Median_Income сохранён.
- Категориальные данные закодированы.
- Полностью дублирующий столбец удалён.
- Пропуски отсутствуют.

**Комментарий**

В целом анализ и предобработка проведены идентично ручной в ЛР1.

Отличие 1 - нулевое значение в строке TotRooms было заменено медианой, а не исключено.

Отличие 2 - использован drop_first=True при Onehot Encoding, что убрало значение `<1H Ocean`. Не критично - все 0 значения задают эту категорию

### Корреляционный анализ

**ПРОМПТ**:

```
Проведи корреляциооный анализ всех признаков. Составь матрицу корреляции.

Проверь, есть ли выбросы в данных и исключи их. Построй гистрограммы распределений значений и boxplot до и после исключения выбросов.

Выбери независимые признаки для дальнейшего построения регрессионных моделей: от 1 признака, от нескольких признаков.

Сформируй обучающую и проверочную части выборки
```
* Включен режим *Think*

**Ответ GPT**

1. Корреляционный анализ

Использован коэффициент корреляции Пирсона. Матрица построена по всем 18 числовым столбцам подготовленного df_gpt, включая целевую переменную и бинарные признаки ocean_proximity.

In [ ]:
# Матрица корреляции Пирсона
corr_matrix = df_gpt.corr(method='pearson')

# Вывод матрицы
display(corr_matrix.round(3))

# Сохранение матрицы
corr_matrix.to_csv(
    'correlation_matrix_gpt.csv',
    sep=';',
    decimal='.'
)

# Корреляция с целевой переменной
target_corr = corr_matrix['Median_Income'].drop(
    'Median_Income'
).sort_values(key=lambda x: abs(x), ascending=False)

display(target_corr)

Наиболее сильная линейная связь с целевой переменной наблюдается у Median_House_Value: коэффициент корреляции составляет 0,688.

In [ ]:
import matplotlib.pyplot as plt

# Тепловая карта полной матрицы корреляций
fig, ax = plt.subplots(figsize=(14, 11))
image = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_xticklabels(corr_matrix.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr_matrix.index, fontsize=8)
ax.set_title('Матрица корреляции Пирсона — до удаления выбросов')
fig.colorbar(image, ax=ax, label='Коэффициент корреляции')
fig.tight_layout()
fig.savefig('correlation_matrix_gpt.png', dpi=160, bbox_inches='tight')
plt.show()

В матрице также обнаружены признаки с высокой взаимной корреляцией:

|Пара признаков|	Корреляция|
|-|-|
|Latitude — Distance_to_SanDiego	| 0,992|
|Distance_to_SanJose — Distance_to_SanFrancisco|	0,990|
|Tot_Bedrooms — Households|	0,980|
|Longitude — Distance_to_SanDiego|	-0,958|
|Longitude — Distance_to_SanFrancisco|	0,955|
|Tot_Rooms — Tot_Bedrooms|	0,930|

Это свидетельствует о потенциальной мультиколлинеарности. Поэтому не рекомендуется без дополнительного отбора включать все признаки в одну регрессионную модель.

2. Выявление и исключение выбросов

Для определения выбросов использован метод межквартильного размаха (IQR).

Границы допустимых значений:

\[
$IQR=Q_3-Q_1$
\]

\[
$X_{\min}=Q_1-1.5\cdot IQR$
\]

\[
$X_{\max}=Q_3+1.5\cdot IQR$
\]

Наблюдение считается выбросом, если хотя бы один непрерывный числовой признак выходит за соответствующие границы.

Бинарные категориальные признаки в проверку не включались.

In [ ]:
# Сохраняем исходную таблицу
df_before_outliers = df_gpt.copy()

# Исключаем бинарные категориальные признаки
binary_cols = [
    col for col in df_gpt.columns
    if col.startswith('ocean_proximity_')
]

continuous_cols = [
    col for col in df_gpt.select_dtypes(
        include=np.number
    ).columns
    if col not in binary_cols
]

# Формирование маски выбросов
outlier_flags = pd.DataFrame(index=df_gpt.index)

for col in continuous_cols:

    Q1 = df_gpt[col].quantile(0.25)
    Q3 = df_gpt[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_flags[col] = (
        (df_gpt[col] < lower) |
        (df_gpt[col] > upper)
    )

# Суммарная маска выбросов
outlier_mask = outlier_flags.any(axis=1)

# Создание таблицы без выбросов
df_gpt_clean = df_gpt.loc[
    ~outlier_mask
].copy().reset_index(drop=True)

print('Исходный размер:', df_gpt.shape)
print('Количество удалённых строк:',
      outlier_mask.sum())

print('Размер после удаления выбросов:',
      df_gpt_clean.shape)

Больше всего выбросов выявлено в следующих признаках:
|Признак|	Количество|
|-|-|
|Distance_to_coast|	2 376|
|Tot_Rooms|	1 287|
|Tot_Bedrooms|	1 284|
|Households|	1 220|
|Population|	1 196|
|Median_House_Value|	1 071|
|Median_Income|	681|

Важно: IQR определяет статистические выбросы, но не обязательно ошибочные данные. В частности, экстремальные расстояния и значения стоимости недвижимости могут соответствовать реальным объектам. В рамках поставленной задачи применена однократная фильтрация по исходным границам IQR.

3. Гистограммы распределений

Построены гистограммы всех непрерывных числовых признаков. Для сравнения использованы одинаковые границы интервалов до и после удаления выбросов.

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(continuous_cols) / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(17, 4 * n_rows)
)

axes = np.asarray(axes).reshape(-1)

for i, col in enumerate(continuous_cols):

    ax = axes[i]

    bins = np.histogram_bin_edges(
        df_before_outliers[col].dropna(),
        bins=35
    )

    ax.hist(
        df_before_outliers[col],
        bins=bins,
        density=True,
        alpha=0.55,
        label='До'
    )

    ax.hist(
        df_gpt_clean[col],
        bins=bins,
        density=True,
        alpha=0.55,
        label='После'
    )

    ax.set_title(col)
    ax.set_ylabel('Плотность')
    ax.legend()

for j in range(len(continuous_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

По графикам видно, что после фильтрации сократились длинные хвосты распределений, особенно у `Tot_Rooms`, `Tot_Bedrooms`, `Population`, `Households` и `Distance_to_coast`.

4. Boxplot до и после удаления выбросов

Boxplot позволяет оценить медиану, межквартильный размах и наличие значений за пределами усов.

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(continuous_cols) / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(17, 4 * n_rows)
)

axes = np.asarray(axes).reshape(-1)

for i, col in enumerate(continuous_cols):

    axes[i].boxplot(
        [
            df_before_outliers[col].dropna(),
            df_gpt_clean[col].dropna()
        ],
        tick_labels=['До', 'После']
    )

    axes[i].set_title(col)
    axes[i].grid(axis='y', alpha=0.25)

for j in range(len(continuous_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

На графиках после удаления выбросов заметно сокращение диапазонов значений для ряда признаков. Отдельные точки за усами могут сохраняться, поскольку фильтрация проводилась однократно по границам исходной таблицы.

5. Выбор независимых признаков

Для дальнейшего построения регрессионных моделей предлагаю два варианта.

Однофакторная регрессия

Используем один признак:

In [ ]:
features_single = ['Median_House_Value']

Основание выбора — максимальная корреляция с целевой переменной.

Многофакторная регрессия

Используем четыре признака:

In [ ]:
features_multi = [
    'Median_House_Value',
    'Tot_Rooms',
    'Median_Age',
    'Distance_to_coast'
]

Выбор основан на сочетании корреляции с целевой переменной и отсутствия сильной взаимной линейной зависимости.

После удаления выбросов корреляции выбранных признаков с Median_Income составляют:

Признак|	Корреляция|
|-|-|
Median_House_Value|	0,609|
Tot_Rooms|	0,285|
Median_Age|	-0,198|
Distance_to_coast|	-0,134|

Дополнительно проверена мультиколлинеарность с использованием VIF (Variance Inflation Factor). Значения для выбранных признаков находятся примерно в диапазоне $1,16–1,24$. Это говорит о низкой мультиколлинеарности выбранного набора.

6. Формирование обучающей и проверочной выборок

Для обучения регрессионных моделей используем разделение:
- 80% — обучающая выборка;
- 20% — проверочная выборка.

Устанавливаем random_state=42 для воспроизводимости результатов.

In [ ]:
from sklearn.model_selection import train_test_split

# Целевая переменная
y = df_gpt_clean['Median_Income']

# Индексы выборок
train_idx, test_idx = train_test_split(
    df_gpt_clean.index,
    test_size=0.2,
    random_state=42
)

# Однофакторная модель
X_train_single = df_gpt_clean.loc[
    train_idx, features_single
]

X_test_single = df_gpt_clean.loc[
    test_idx, features_single
]

# Многофакторная модель
X_train_multi = df_gpt_clean.loc[
    train_idx, features_multi
]

X_test_multi = df_gpt_clean.loc[
    test_idx, features_multi
]

# Обучающая и проверочная целевые переменные
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

# Проверка размеров
print('Однофакторная модель:')
print('X_train:', X_train_single.shape)
print('X_test:', X_test_single.shape)

print('\nМногофакторная модель:')
print('X_train:', X_train_multi.shape)
print('X_test:', X_test_multi.shape)

print('\nЦелевая переменная:')
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

8. Итог и готовность к построению моделей

На следующем этапе можно переходить к обучению регрессионных моделей.

Подготовлены два варианта входных данных:

|Модель|	Независимые признаки|	Целевая переменная|
|-|-|-|
|Однофакторная|	Median_House_Value	|Median_Income
|Многофакторная|	Median_House_Value, Tot_Rooms, Median_Age, Distance_to_coast	|Median_Income

**Методологическое замечание**: границы IQR в этой работе рассчитаны до разделения на обучение и проверку, согласно выбранной последовательности обработки. Для строгой оценки качества будущих моделей желательно рассчитывать параметры предобработки только на обучающей части и применять их к проверочной.

Также важно придерживаться выбранного Median_Income в качестве TARGET: именно он обозначен целевой переменной в описании приложенного датасета.

**Комментарий**

Для построения тепловой карты корреляции признаков использовался matplotlib, а не seaborn, как в ЛР1 - отсюда отсутствие текстовых меток на пересечении признаков. Сам код для генерации ихображения был вручную взят из ipynb, который модель использует для *"своего анализа"*

Дополнительно была учтена межпризнаковая корреляция, чего в ЛР1 явно сделано не было. Однако это не повлияло на выбор признаков ни для единичного случая, ни для регрессии по множеству признаков.

Выбросы убирались автоматически по результатам расчета, а не вручную, как в ЛР1. Следствие - разница в выборках 15404 и 16361 для РГР и ЛР1 соответственно.

### Построение линейной регрессионной модели от ОДНОГО признака

**ПРОМПТ**:

```
Построй и проанализируй линейную регрессионную модель (tf/keras) от одного признака.
Для анализа построй график разброса тренировочных и проверочных данных (целевой y и независимый признак x):
1. Отложи на нем полученную линию регрессии. Выведи формулу y=f(x) с полученными весами.
2. Построй плотность распределения и гистограмму ошибок. 
3. Построй диаграммы рассеяния вычисленных значений (pred/true) и диаграмму рассеяния квадрата ошибки.
4. Получи и выведи метрики MAE/MSE/RMSE/R2

Вынеси пункты 2-4 в отдельные функции для дальнейшего использования для других моделей.

Для метрик используй scikit-learn.
```

* Включен редим Think

**Ответ GPT**

1. Построение модели Keras

Используется линейная нейронная сеть с одним выходным нейроном:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

In [ ]:
scaler = StandardScaler()
X_train_single_scaled = scaler.fit_transform(X_train_single).astype("float32")
X_test_single_scaled = scaler.transform(X_test_single).astype("float32")

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(1,)),
    keras.layers.Dense(1, activation=None)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.03),
    loss='mse'
)

history = model.fit(
    X_train_single_scaled,
    y_train,
    validation_data=(X_test_single_scaled, y_test),
    epochs=80,
    batch_size=512,
    verbose=0
)

print("Последняя ошибка train:", history.history["loss"][-1])
print("Последняя ошибка validation:", history.history["val_loss"][-1])

Перед обучением выполнено масштабирование Median_House_Value посредством StandardScaler. Масштабирование обучено только на тренировочной выборке.

2. Формула линейной регрессии

Поскольку входной признак был стандартизирован, сначала извлекаются вес w и смещение b в пространстве стандартизированного признака. Затем вычисляются коэффициенты для исходного Median_House_Value:

\[
$\hat{y} = b_0 + b_1 x$
\]

In [ ]:
w_scaled = float(model.get_weights()[0][0, 0])
b_scaled = float(model.get_weights()[1][0])

coef = w_scaled / float(scaler.scale_[0])
intercept = b_scaled - w_scaled * float(scaler.mean_[0]) / float(scaler.scale_[0])

print(f"Median_Income = {intercept:.8f} + ({coef:.10f}) * Median_House_Value")

# Предсказания
pred_train = model.predict(X_train_single_scaled, verbose=0).reshape(-1)
pred_test = model.predict(X_test_single_scaled, verbose=0).reshape(-1)

# Диаграмма рассеяния train/test и линия регрессии
x_line = np.linspace(
    min(X_train_single[features_single[0]].min(), X_test_single[features_single[0]].min()),
    max(X_train_single[features_single[0]].max(), X_test_single[features_single[0]].max()),
    300
)
y_line = intercept + coef * x_line

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_train_single[features_single[0]], y_train, s=13, alpha=0.35, label="Обучающая выборка")
ax.scatter(X_test_single[features_single[0]], y_test, s=16, alpha=0.50, label="Проверочная выборка")
ax.plot(x_line, y_line, linewidth=2.5, label="Линия регрессии (обучена на train)")
ax.set_xlabel("Median_House_Value, USD")
ax.set_ylabel("Median_Income, 10k USD")
ax.set_title("Однофакторная линейная регрессия")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

Например, увеличение стоимости жилья на 100 000 долларов соответствует увеличению прогнозируемого Median_Income примерно на 0,976 (в единицах 10 тысяч долларов).

3. Распределение ошибок

Ошибка рассчитана как разница между истинным и предсказанным значениями:
\[
$e=y_{\text{true}}-y_{\text{pred}}$
\]

Реализована отдельная функция `plot_error_distribution()`.

Она строит гистограмму ошибок и оценку плотности распределения KDE.

In [ ]:
def plot_error_distribution(y_true, y_pred, title="Распределение ошибок"):
    """Показывает гистограмму и KDE плотность ошибок (y_true - y_pred)."""
    import scipy.stats as st

    errors = np.asarray(y_true).reshape(-1) - np.asarray(y_pred).reshape(-1)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(errors, bins=40, density=True, alpha=0.55,
            edgecolor="white", label="Гистограмма ошибок")

    if len(errors) > 1 and np.std(errors) > 0:
        xs = np.linspace(errors.min(), errors.max(), 400)
        kde = st.gaussian_kde(errors)
        ax.plot(xs, kde(xs), linewidth=2, label="Оценка плотности (KDE)")

    ax.axvline(0, linestyle="--", linewidth=1.5, label="Нулевая ошибка")
    ax.set_xlabel("Ошибка (y_true - y_pred)")
    ax.set_ylabel("Плотность")
    ax.set_title(title)
    ax.grid(alpha=0.2)
    ax.legend()
    plt.tight_layout()
    return fig, ax

fig, ax = plot_error_distribution(
    y_test, pred_test,
    title="Распределение ошибок — проверочная выборка"
)
plt.show()

*Наблюдение*: распределение сосредоточено около нулевой ошибки, но имеет достаточно широкие хвосты. Это указывает на наличие значительных ошибок для отдельных наблюдений.

4. Диаграммы рассеяния

Реализована функция `plot_prediction_diagnostics()`.

Она строит два графика:
- Предсказанные значения pred относительно истинных true, с линией идеального совпадения.
- Квадрат ошибки относительно предсказанного значения.

In [ ]:
def plot_prediction_diagnostics(y_true, y_pred, title="Диагностика предсказаний"):
    """Строит pred/true и квадрат ошибки относительно pred."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    squared_error = (y_true - y_pred) ** 2

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    axes[0].scatter(y_true, y_pred, s=16, alpha=0.45)
    axes[0].plot([lo, hi], [lo, hi], linestyle="--", linewidth=2,
                 label="Идеальное совпадение: pred = true")
    axes[0].set_xlabel("Истинные значения (true)")
    axes[0].set_ylabel("Предсказанные значения (pred)")
    axes[0].set_title("Предсказанные и истинные значения")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    axes[1].scatter(y_pred, squared_error, s=16, alpha=0.45)
    axes[1].set_xlabel("Предсказанное значение (pred)")
    axes[1].set_ylabel("Квадрат ошибки (y_true - y_pred)^2")
    axes[1].set_title("Квадрат ошибки относительно pred")
    axes[1].grid(alpha=0.2)

    fig.suptitle(title)
    plt.tight_layout()
    return fig, axes

fig, axes = plot_prediction_diagnostics(
    y_test, pred_test,
    title="Диагностика — проверочная выборка"
)
plt.show()

На первой диаграмме заметен значительный разброс относительно линии идеального совпадения. Модель недостаточно хорошо описывает индивидуальные значения целевой переменной.

На второй диаграмме видны отдельные наблюдения с большими квадратами ошибок.

5. Метрики качества модели

Для расчёта использована библиотека scikit-learn.

Создана универсальная функция `calculate_regression_metrics()`.

In [ ]:
def calculate_regression_metrics(y_true, y_pred):
    """Возвращает MAE, MSE, RMSE и R2 с использованием scikit-learn."""
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }

metrics = pd.DataFrame(
    [
        calculate_regression_metrics(y_train, pred_train),
        calculate_regression_metrics(y_test, pred_test)
    ],
    index=["Train", "Test"]
)

display(metrics.round(5))

Все метрики ошибок выражены в единицах целевой переменной (для MAE и RMSE — примерно 9 200 и 11 424 долларов соответственно на проверочной выборке).

Интерпретация:
- MAE показывает среднюю абсолютную ошибку прогнозирования.
- MSE сильнее штрафует большие ошибки.
- RMSE показывает среднюю величину ошибки в исходных единицах целевой переменной.
- R² = 0,3695 означает, что модель объясняет приблизительно 37% вариации Median_Income на проверочной выборке.

Значения метрик на обучающей и проверочной выборках близки. Выраженных признаков переобучения не наблюдается, однако качество однофакторной модели пока ограничено.

**Комментарий**

Вручную были добавлены импорты библиотек. Вручную изменены названия переменных для входных данных (с X_train на X_train_single, FEATURES на features_single[0] и т.п.).

Вместо linear_model.LinearRegression, как в ЛР1, GPT использует Sequential-модель.

Полученные графики идентичны ручным ЛР1. Внешний вид диаграмм совпадает. Сравнительно ЛР1 метрика $R^2$ меньше на 0.007 - метрики идентичны

In [ ]:
models_r2_data[0][0] = r2_score(y_test, pred_test)

### Построение линейной регрессионной модели от нескольких признаков (переменных) в соответствии с рекомендацией от LLM и выполнение анализа значимости и вклада выбранных признаков.

**ПРОМПТ**:

```
Построй линейную регрессионную модель от нескольких признаков и выполни анализ значимости и вклада выбранных признаков. Модель без скрытых слоев.
Для анализа используй ранее введенные функции.
Выведи формулу y=f(X)
```

* Включен режим Think

**Ответ GPT**

1. Архитектура модели

Модель непосредственно связывает четыре входных признака с одним выходом:

\[
$\hat y=b+w_1x_1+w_2x_2+w_3x_3+w_4x_4$
\]

Скрытых слоёв и нелинейных функций активации нет. Для улучшения численной устойчивости входные признаки стандартизированы; итоговая формула ниже приведена в исходных единицах.

StandardScaler обучается исключительно на тренировочных данных. Проверочная выборка преобразуется теми же параметрами. При выводе итоговой формулы коэффициенты пересчитываются к исходным единицам признаков.

In [ ]:
scaler_multi = StandardScaler()
X_train_scaled_multi = scaler_multi.fit_transform(X_train_multi).astype("float32")
X_test_scaled_multi = scaler_multi.transform(X_test_multi).astype("float32")

2. Построение и обучение модели без скрытых слоёв

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(len(features_multi),), name="four_features_input"),
    keras.layers.Dense(1, activation=None, name="linear_output")
], name="multi_feature_linear_regression")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.1),
    loss="mse"
)

history = model.fit(
    X_train_scaled_multi,
    y_train,
    epochs=500,
    batch_size=len(X_train_multi),  # full-batch gradient update
    shuffle=False,
    verbose=0
)

print("Последняя ошибка обучения:", history.history["loss"][-1])
model.summary()

3. Получение весов и формулы в исходных единицах

Keras обучает линейную модель на стандартизированных входах. Если $z_j = (x_j - mean_j) / scale_j$, а внутреннее уравнение модели $y = b_{scaled} + sum(w_{scaled_j} * z_j)$, то после обратного преобразования получается формула $y = intercept + sum(coef_j * x_j)$.

In [ ]:
weights_scaled = model.get_weights()[0].reshape(-1).astype(float)
bias_scaled = float(model.get_weights()[1][0])

coef = weights_scaled / scaler.scale_
intercept = bias_scaled - np.sum(weights_scaled * scaler.mean_ / scaler.scale_)

coef_series = pd.Series(coef, index=features_multi, name="Коэффициент")
print(f"Median_Income = {intercept:.10f}")
for name, value in coef_series.items():
    sign = "+" if value >= 0 else "-"
    print(f"  {sign} {abs(value):.12g} * {name}")

pred_train = model.predict(X_train_scaled_multi, verbose=0).reshape(-1)
pred_test = model.predict(X_test_scaled_multi, verbose=0).reshape(-1)

Положительные коэффициенты означают положительную связь с прогнозом при фиксированных остальных признаках. Отрицательный коэффициент Median_Age означает обратную связь.

4. Повторное использование ранее введённых функций анализа

4.1. Распределение ошибок на проверочной выборке

In [ ]:
fig_error, ax_error = plot_error_distribution(
    y_test, pred_test,
    title="Многофакторная модель: распределение ошибок (test)"
)
fig_error.savefig("multi_feature_error_distribution_gpt.png", dpi=160, bbox_inches="tight")
plt.show()

4.2. Диаграммы pred/true и квадрата ошибки

In [ ]:
fig_diag, axes = plot_prediction_diagnostics(
    y_test, pred_test,
    title="Многофакторная модель: диагностика (test)"
)
fig_diag.savefig("multi_feature_prediction_diagnostics_gpt.png", dpi=160, bbox_inches="tight")
plt.show()

5. Метрики качества (scikit-learn)

In [ ]:
metrics = pd.DataFrame(
    [
        calculate_regression_metrics(y_train, pred_train),
        calculate_regression_metrics(y_test, pred_test)
    ],
    index=["Train", "Test"]
)

display(metrics.round(6))
metrics.to_csv("multi_feature_metrics_gpt.csv", sep=";", decimal=".")

На проверочной выборке R² составил 0,4356. Модель объясняет около 43,56% вариации целевой переменной.

Сравнение с однофакторной моделью:
|Метрика Test	|1 признак|	4 признака|
|-|-|-|
|MAE|	0,9173|	0,8655|
|RMSE|	1,1424|	1,0808|
|R²|	0,3695|	0,4356|

Добавление трёх признаков:
- увеличило R² на 0,0661;
- уменьшило MAE на 5,65%;
- уменьшило RMSE на 5,39%.

In [ ]:
models_r2_data[0][1] = r2_score(y_test, pred_test)

6. Статистическая значимость коэффициентов

Keras обучает параметры, но не рассчитывает p-value и доверительные интервалы. Для вывода об их статистической значимости оценивается OLS той же линейной спецификации только на тренировочной выборке. Стандартные ошибки HC3 устойчивы к гетероскедастичности. Предикторы стандартизированы; коэффициент в этой таблице показывает изменение Median_Income при увеличении соответствующего признака на одно стандартное отклонение. Масштабирование не меняет p-value линейного коэффициента.

Для оценки статистической значимости использована дополнительная OLS-модель с робастными стандартными ошибками HC3.

In [ ]:
import statsmodels.api as sm

X_train_scaled_df_multi = pd.DataFrame(
    X_train_scaled_multi, columns=features_multi, index=X_train_multi.index
)
ols_model = sm.OLS(
    y_train,
    sm.add_constant(X_train_scaled_df_multi)
).fit(cov_type="HC3")

conf_int = ols_model.conf_int()
rows = []
for feature in features_multi:
    rows.append({
        "Признак": feature,
        "Коэффициент на 1 SD X": ols_model.params[feature],
        "Robust SE HC3": ols_model.bse[feature],
        "z": ols_model.tvalues[feature],
        "p-value": ols_model.pvalues[feature],
        "95% CI нижняя": conf_int.loc[feature, 0],
        "95% CI верхняя": conf_int.loc[feature, 1]
    })

significance_table = pd.DataFrame(rows).set_index("Признак")
display(significance_table.style.format({
    "Коэффициент на 1 SD X": "{:.5f}",
    "Robust SE HC3": "{:.5f}",
    "z": "{:.3f}",
    "p-value": "{:.3e}",
    "95% CI нижняя": "{:.5f}",
    "95% CI верхняя": "{:.5f}"
}))
significance_table.to_csv("multi_feature_significance_gpt.csv", sep=";", decimal=".")

print("R² OLS на train:", round(ols_model.rsquared, 6))
print("Adjusted R²:", round(ols_model.rsquared_adj, 6))
print("F-statistic:", round(float(ols_model.fvalue), 3))
print("p-value общей модели:", ols_model.f_pvalue)

Все четыре коэффициента статистически значимы при уровне значимости 0,05.

Общая модель также статистически значима: F-статистика ≈ 1905,61, p-value меньше 0,001.

7. Анализ вклада отдельных признаков

Для сопоставления влияния признаков рассчитываются стандартизированные коэффициенты beta: коэффициент при стандартизированном X делится на стандартное отклонение y. Их абсолютные значения нормируются до 100% — это относительный рейтинг по силе линейной связи в многомерной модели, а не строгое аддитивное разложение R².

In [ ]:
y_std = float(np.std(y_train, ddof=0))
standardized_beta = ols_model.params[features_multi] / y_std
relative_contribution = (
    standardized_beta.abs() / standardized_beta.abs().sum() * 100
)

contribution_table = pd.DataFrame({
    "Стандартизированный beta": standardized_beta,
    "Относительный вклад, %": relative_contribution
}).sort_values("Относительный вклад, %", ascending=False)

display(contribution_table.round(4))
contribution_table.to_csv("multi_feature_contribution_gpt.csv", sep=";", decimal=".")

fig, ax = plt.subplots(figsize=(9, 5))
contribution_table["Относительный вклад, %"].sort_values().plot.barh(ax=ax)
ax.set_xlabel("Относительный вклад по |standardized beta|, %")
ax.set_ylabel("Признак")
ax.set_title("Сравнение стандартизированных коэффициентов")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
fig.savefig("multi_feature_standardized_contribution_gpt.png", dpi=160, bbox_inches="tight")
plt.show()

8. Перестановочная важность на проверочной выборке

Дополнительный анализ вклада — permutation importance: значения одного признака случайно перемешиваются в проверочной выборке, остальные признаки сохраняются. Измеряется среднее снижение R² по 30 перестановкам. Чем больше снижение R², тем сильнее модель зависит от этого признака. Это оценка важности для данной модели на данной проверочной выборке, не причинный эффект.

In [ ]:
rng = np.random.default_rng(42)
base_r2 = r2_score(y_test, pred_test)
permutation_rows = []
N_REPEATS = 30

for j, feature in enumerate(features_multi):
    score_drops = []
    for _ in range(N_REPEATS):
        X_permuted = X_test_scaled_multi.copy()
        X_permuted[:, j] = rng.permutation(X_permuted[:, j])
        perm_pred = model.predict(X_permuted, verbose=0).reshape(-1)
        score_drops.append(base_r2 - r2_score(y_test, perm_pred))

    permutation_rows.append({
        "Признак": feature,
        "Среднее снижение R²": np.mean(score_drops),
        "Стандартное отклонение": np.std(score_drops, ddof=1)
    })

permutation_table = pd.DataFrame(permutation_rows).set_index("Признак")
permutation_table = permutation_table.sort_values("Среднее снижение R²", ascending=False)
display(permutation_table.round(5))
permutation_table.to_csv("multi_feature_permutation_importance_gpt.csv", sep=";", decimal=".")

fig, ax = plt.subplots(figsize=(9, 5))
permutation_table["Среднее снижение R²"].sort_values().plot.barh(ax=ax, xerr=permutation_table.loc[permutation_table["Среднее снижение R²"].sort_values().index, "Стандартное отклонение"])
ax.set_xlabel("Среднее снижение R² после перемешивания")
ax.set_ylabel("Признак")
ax.set_title("Перестановочная важность признаков (test, 30 повторов)")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
fig.savefig("multi_feature_permutation_importance_gpt.png", dpi=160, bbox_inches="tight")
plt.show()

Дополнительно рассчитана перестановочная важность на проверочной выборке. При случайном перемешивании значений Median_House_Value качество модели резко ухудшается. Это подтверждает его доминирующую роль в прогнозировании.

9. Проверка мультиколлинеарности

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_table = pd.DataFrame({
    "Признак": features_multi,
    "VIF": [variance_inflation_factor(X_train_scaled_df_multi.astype(float), i)
            for i in range(len(features_multi))]
}).set_index("Признак")
display(vif_table.round(3))


Значения VIF всех четырёх признаков находятся в диапазоне 1,17–1,24. Следовательно, выраженной мультиколлинеарности в выбранном наборе нет.

**Комментарий**

Также вручную изменены названия переменных исходного датасета. Также не используется linear_model.LinearRegression. Однако выходные метрики идентичны.
Дополнительно GPT провела статистический анализ значимости признаков, перестановочная важность, проверена мултиколлинеарность, чего в ЛР1 сделано не было

### Построение регрессионной модели на основе многослойного персептрона

**ПРОМПТ**:

```
Построй регрессионную модель на основе многослойного персептрона множества признаков. Построй график изменения ошибки от эпох обучения. Проведи аналогичный анализ точности модели: pred и true, квадрат ошибки, метрики. Оцени вклад каждого признака.
```

* Включен режим think

**Ответ GPT**

1. Подготовка данных и архитектура MLP

Числовые признаки и целевая переменная стандартизируются. StandardScaler обучается только на внутренней обучающей части X_fit, y_fit. Архитектура содержит два скрытых полносвязных слоя (64 и 32 нейрона, ReLU) и один линейный выход. Это регрессионная MLP, а не линейная модель без скрытых слоёв.

In [ ]:
X_train_all = df_gpt_clean[features_multi].astype('float32')
y_train_all = df_gpt_clean[target_col].astype('float32')

# Внутренняя validation-выборка формируется только из train.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_all, y_train_all,
    test_size=0.20,
    random_state=42
)

In [ ]:
from keras import layers, regularizers

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_fit_s = x_scaler.fit_transform(X_fit).astype('float32')
X_val_s = x_scaler.transform(X_val).astype('float32')
X_train_all_s = x_scaler.transform(X_train_all).astype('float32')
X_test_s = x_scaler.transform(X_test_multi).astype('float32')

y_fit_s = y_scaler.fit_transform(y_fit.to_numpy().reshape(-1, 1)).astype('float32')
y_val_s = y_scaler.transform(y_val.to_numpy().reshape(-1, 1)).astype('float32')

model = keras.Sequential([
    layers.Input(shape=(len(features_multi),), name='features'),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.10),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dense(1, activation='linear', name='median_income')
], name='california_housing_mlp')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

model.summary()


2. Обучение и изменение ошибки по эпохам

Для остановки обучения используются EarlyStopping и ReduceLROnPlateau. Контрольная выборка test не участвует в выборе эпохи.

In [ ]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=25,
        min_delta=0.0,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-5,
        verbose=1
    )
]

history = model.fit(
    X_fit_s, y_fit_s,
    validation_data=(X_val_s, y_val_s),
    epochs=200,
    batch_size=2048,
    callbacks=callbacks,
    verbose=0
)

best_epoch = int(np.argmin(history.history['val_loss']) + 1)
print(f'Фактическое число эпох: {len(history.history["loss"])}')
print(f'Лучшая эпоха по val_loss: {best_epoch}')

plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train MSE (стандартизированная y)')
plt.plot(history.history['val_loss'], label='Validation MSE (стандартизированная y)')
plt.axvline(best_epoch - 1, color='gray', linestyle='--', label=f'Лучшая эпоха: {best_epoch}')
plt.xlabel('Эпоха')
plt.ylabel('MSE loss')
plt.title('Изменение ошибки в процессе обучения MLP')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('mlp_training_history_gpt.png', dpi=160, bbox_inches='tight')
plt.show()


**ПРИМ: не соответствует рисунку**

Результаты обучения:
- Выполнено 200 эпох.
- Минимальная ошибка на validation достигнута на 200-й эпохе.
- В ходе обучения дважды уменьшался learning rate.
- Ошибка на обучающей выборке постепенно снижалась.
- Проверочная ошибка также снижалась, но медленнее.
Разрыв между тренировочной и проверочной ошибкой небольшой. Выраженного переобучения по графику не наблюдается.

3. Прогнозы и метрики

Прогнозы преобразуются обратно в исходный масштаб Median_Income. Для оценки используются MAE, MSE, RMSE и R² из scikit-learn.

In [ ]:
def predict_original_scale(model, X_scaled, scaler_y):
    pred_scaled = model.predict(X_scaled, verbose=0)
    return scaler_y.inverse_transform(pred_scaled).ravel()

pred_train = predict_original_scale(model, X_train_all_s, y_scaler)
pred_test = predict_original_scale(model, X_test_s, y_scaler)
y_train_np = y_train_all.to_numpy()
y_test_np = y_test.to_numpy()


def calculate_regression_metrics(y_true, y_pred):
    """Возвращает основные метрики регрессии в исходном масштабе y."""
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mean_squared_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

metrics_mlp = pd.DataFrame({
    'Train': calculate_regression_metrics(y_train_np, pred_train),
    'Test': calculate_regression_metrics(y_test_np, pred_test)
}).T

print(metrics_mlp.round(6))
metrics_mlp.to_csv('mlp_metrics_gpt.csv', sep=';', decimal='.')


Коэффициент детерминации на проверочной выборке
0,4738
Модель объясняет приблизительно 47,38% вариации целевой переменной.

MAE = 0,8306 означает, что средняя абсолютная ошибка составляет около 8 306 долларов, поскольку Median_Income измеряется в десятках тысяч долларов.

In [ ]:
models_r2_data[0][2] = r2_score(y_test_np, pred_test)

4. Диагностика ошибок

Функции построения распределения ошибок и диаграмм pred/true и квадрата ошибки определены отдельно, чтобы их можно было повторно использовать для других моделей

In [ ]:
from scipy.stats import gaussian_kde


def plot_error_distribution(y_true, y_pred, title='Распределение ошибок', save_path=None):
    """Гистограмма и KDE ошибок e = y_true - y_pred."""
    errors = np.asarray(y_true).ravel() - np.asarray(y_pred).ravel()
    plt.figure(figsize=(9, 5))
    plt.hist(errors, bins=40, density=True, alpha=0.60, edgecolor='white', label='Гистограмма')
    if np.std(errors) > 0:
        grid = np.linspace(errors.min(), errors.max(), 400)
        plt.plot(grid, gaussian_kde(errors)(grid), linewidth=2, label='KDE')
    plt.axvline(0, color='black', linestyle='--', linewidth=1, label='Нулевая ошибка')
    plt.xlabel('Ошибка: true − pred')
    plt.ylabel('Плотность')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=160, bbox_inches='tight')
    plt.show()


def plot_prediction_diagnostics(y_true, y_pred, title='Диагностика прогнозов', save_path=None):
    """Scatter pred vs true и квадрат ошибки относительно предсказания."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    errors_sq = (y_true - y_pred) ** 2
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(y_true, y_pred, alpha=0.35, s=12)
    axes[0].plot([lo, hi], [lo, hi], 'r--', label='Идеальный прогноз')
    axes[0].set_xlabel('Истинное значение (true)')
    axes[0].set_ylabel('Предсказание (pred)')
    axes[0].set_title('Предсказанные и истинные значения')
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    axes[1].scatter(y_pred, errors_sq, alpha=0.35, s=12)
    axes[1].set_xlabel('Предсказанное значение (pred)')
    axes[1].set_ylabel('(true − pred)²')
    axes[1].set_title('Квадрат ошибки')
    axes[1].grid(alpha=0.2)

    fig.suptitle(title)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=160, bbox_inches='tight')
    plt.show()

In [ ]:
plot_error_distribution(y_test_np, pred_test, 'Распределение ошибок MLP на test', 'mlp_error_distribution_gpt.png')
plot_prediction_diagnostics(y_test_np, pred_test, 'MLP: диагностика на test', 'mlp_prediction_diagnostics_gpt.png')

На первом графике точки располагаются относительно линии идеального прогноза. Видно, что модель улавливает зависимость, однако сохраняется разброс предсказаний.

На втором графике показаны квадраты ошибок. Присутствуют отдельные наблюдения с большими ошибками.

Ошибки рассчитаны как:
\[
$e=y_{\text{true}}-y_{\text{pred}}$
\]

Распределение сосредоточено около нуля. Есть отдельные большие положительные и отрицательные ошибки.

5. Вклад признаков: permutation importance

Для нейросети веса скрытых слоёв нельзя корректно интерпретировать как прямые коэффициенты линейной модели. Поэтому вклад оценивается перестановочной важностью на отложенной проверочной выборке: по одному признаку случайно перемешиваются значения, после чего измеряется снижение R². Чем больше снижение, тем важнее признак для прогноза модели.

Проценты ниже нормируются по сумме положительных снижений R². Это относительная оценка именно для данной модели и выборки, а не причинный эффект.

In [ ]:
from sklearn.metrics import r2_score

base_r2 = r2_score(y_test_np, pred_test)
rng = np.random.default_rng(42)
N_REPEATS = 20
importance_rows = []

for j, feature in enumerate(features_multi):
    drops = []
    for _ in range(N_REPEATS):
        X_perm = X_test_s.copy()
        X_perm[:, j] = rng.permutation(X_perm[:, j])
        pred_perm = predict_original_scale(model, X_perm, y_scaler)
        drops.append(base_r2 - r2_score(y_test_np, pred_perm))
    importance_rows.append({
        'Признак': feature,
        'Среднее снижение R2': float(np.mean(drops)),
        'Стандартное отклонение': float(np.std(drops, ddof=1))
    })

importance_df = pd.DataFrame(importance_rows).sort_values(
    'Среднее снижение R2', ascending=False
).reset_index(drop=True)
positive_total = importance_df['Среднее снижение R2'].clip(lower=0).sum()
importance_df['Относительный вклад, %'] = (
    100 * importance_df['Среднее снижение R2'].clip(lower=0) / positive_total
    if positive_total > 0 else 0.0
)

display(importance_df.round(5))
importance_df.to_csv('mlp_feature_importance_gpt.csv', sep=';', decimal='.', index=False)

plt.figure(figsize=(9, 5))
plt.barh(
    importance_df['Признак'][::-1],
    importance_df['Среднее снижение R2'][::-1],
    xerr=importance_df['Стандартное отклонение'][::-1],
    capsize=4
)
plt.xlabel('Снижение R² после перемешивания признака')
plt.title('Permutation importance признаков MLP (test)')
plt.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig('mlp_feature_importance_gpt.png', dpi=160, bbox_inches='tight')
plt.show()


Интерпретация:

1. Median_House_Value — основной фактор прогнозирования, его вклад значительно превосходит остальные.
2. Median_Age занимает второе место.
3. Distance_to_coast оказывает дополнительное влияние.
4. Tot_Rooms имеет наименьшую перестановочную важность среди выбранных признаков.

Относительные вклады рассчитаны нормированием положительных снижений R². Они характеризуют важность признаков внутри этой модели, а не причинное влияние на доход.

6. Общий вывод

Многослойный персептрон показал лучшее качество прогнозирования Median_Income по сравнению с однофакторной и многофакторной линейными регрессиями.

Получено R² = 0,4744 на проверочной выборке. Наибольший вклад в прогнозирование вносит Median_House_Value, однако нейросеть учитывает и нелинейные зависимости между несколькими признаками.

Результат: MLP можно выбрать в качестве лучшей из трёх построенных моделей по рассмотренным метрикам. При этом качество пока оставляет пространство для улучшения: можно исследовать архитектуру сети, подобрать гиперпараметры и проверить дополнительные признаки.

**Комментарий**:

В ЛР3 (на 2025) наилучший полученный перцептрон состоял из 4 скрытых слоев - $1000 \to 500 \to 200 \to 10$ (605,721 параметров) и достигал $R^2=0.534$. LLM же выбрал модель на 2 скрытых слоя с dropout и колбэками на уменьшение `learning_rate` и раннюю остановку (2,433 параметра). Полученная LLM модель хуже по метрике $R^2$ на 0.06, но обучалась столько же эпох. При этом batch_size в ЛР3 подбирался экспериментально, GPT просто указал 2048 - $\frac{1}{6}$ выборки, большие обобщения в сравнение с 64. Однако это сильно ускорило процесс обучения

### Сравнение с ЛР3

**ПРОМПТ**

```
Я ранее строил MLP-модель с другими признаками:

features = ['Median_House_Value', 'Distance_to_coast', 'Tot_Rooms', 'Median_Age', 'Population']

def initNewModel(size=5):
    global totalHistoryLossTrain
    totalHistoryLossTrain=[]
    global totalHistoryLossTest
    totalHistoryLossTest=[]

    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(shape=(size,)))

    model.add(tf.keras.layers.Dense(units=1000,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=500,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=200,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=10,  activation=tf.keras.activations.relu))

    model.add(tf.keras.layers.Dense(units=1,  activation=None))

    fLoss=tf.keras.losses.MeanSquaredError()
    fOptimizer=tf.keras.optimizers.Adam(learning_rate=0.01)
    fMetricList  = [tf.keras.losses.MeanSquaredError(), tf.keras.losses.MeanAbsoluteError()]
    model.compile(
        loss      =fLoss,
        optimizer =fOptimizer,
        metrics   = fMetricList
    )
    return model

И проводил эксперименты
Эксперименты с batch_size:

Признаки  Структура ЭпохОбучения  Размер  R2  RMSE
0 [Median_House_Value, Distance_to_coast, Tot_Ro... 5вх-> 4 ->1вых  100 64  0.528724  0.131514
1 [Median_House_Value, Distance_to_coast, Tot_Ro... 5вх-> 4 ->1вых  100 128 0.509933  0.134111
2 [Median_House_Value, Distance_to_coast, Tot_Ro... 5вх-> 4 ->1вых  100 256 0.534347  0.130727
3 [Median_House_Value, Distance_to_coast, Tot_Ro... 5вх-> 4 ->1вых  100 1024  0.527885  0.131631

Эксперименты с epoch:

Признаки  Структура ЭпохОбучения  R2  RMSE
0 [Median_House_Value, Distance_to_coast, Tot_Ro... 5вх-> 4 ->1вых  20  0.473984  0.138942
1 [Median_House_Value, Distance_to_coast, Tot_Ro... 5вх-> 4 ->1вых  80  0.511734  0.133864
2 [Median_House_Value, Distance_to_coast, Tot_Ro... 5вх-> 4 ->1вых  160 0.531746  0.131092
3 [Median_House_Value, Distance_to_coast, Tot_Ro... 5вх-> 4 ->1вых  500 0.480007  0.138145

Сравни свои результаты с моими, проинтерпритируй разницу структуры MLP / параметров обучения и их влияние на метрики.
```

* Включен Think

**Ответ GPT**

Твои эксперименты показывают очень интересный результат: **твоя MLP достигает R² ≈ 0,534, тогда как построенная мной модель — R² = 0,4744**. Разница достаточно заметная. Причём она может объясняться не только увеличением количества нейронов, но и другими признаками, размером batch, количеством эпох и скоростью обучения.

Разберём результаты с точки зрения архитектуры нейронной сети и оптимизации.

#### **1. Сравнение архитектур MLP**

Сначала сравним непосредственно две архитектуры.

| Характеристика     | Твоя MLP        | Моя MLP |
| ------------------ | --------------- | ------- |
| Входных признаков  | 5               | 4       |
| Скрытых слоёв      | 4               | 2       |
| Нейронов в слоях   | 1000-500-200-10 | 64-32   |
| Выходных нейронов  | 1               | 1       |
| Функция активации  | ReLU            | ReLU    |
| Выходная активация | Linear          | Linear  |
| Dropout            | Нет             | 0,10    |
| L2-регуляризация   | Нет             | 0,0001  |

Важное различие — количество обучаемых параметров.

Для твоей архитектуры:

$\begin{aligned} N=(5\cdot1000+1000)\\+(1000\cdot500+500)\\+(500\cdot200+200)\\+(200\cdot10+10)\\+(10\cdot1+1) \end{aligned}$

$\boxed{N=608721}$

Для моей архитектуры:

$N=(4\cdot64+64)+(64\cdot32+32)+(32+1)$

$\boxed{N=2433}$

Твоя модель - **608 721** Обучаемых параметров

Моя модель - **2 433** Обучаемых параметров

Твоя сеть имеет примерно в **250 раз больше обучаемых параметров**.

Это принципиальная разница.

**Что означает такая разница?**

Твоя модель обладает существенно большей выразительной способностью. Она может находить более сложные нелинейные зависимости и взаимодействия признаков.

Например, влияние стоимости жилья на доход может отличаться в зависимости от:

- численности населения;
- расстояния до побережья;
- количества комнат;
- возраста недвижимости.

Большая сеть способна моделировать такие взаимодействия.

Но за это приходится платить повышенным риском переобучения и большей чувствительностью к гиперпараметрам.

Моя модель компактнее, имеет L2-регуляризацию и Dropout. Она ограничивает сложность модели, но может оказаться недостаточно гибкой для выявления сложных зависимостей.

#### **2. Сравнение использованных признаков**

Здесь также есть важное отличие.

**Твой набор:**

```
features = [    'Median_House_Value',    'Distance_to_coast',    'Tot_Rooms',    'Median_Age',    'Population']
```

**Мой набор:**

```
features = [    'Median_House_Value',    'Tot_Rooms',    'Median_Age',    'Distance_to_coast']
```

Единственное отличие — `Population`.

Это вполне обоснованный дополнительный признак.

Численность населения может содержать информацию о плотности заселения, типе населённого пункта и характеристиках жилого района.

При этом `Population` может быть коррелирован с `Tot_Rooms`, поэтому его дополнительную ценность лучше оценивать не только по корреляции, но и по permutation importance.

**Важный момент:** поскольку одновременно изменились архитектура и набор признаков, нельзя утверждать, что улучшение R² произошло исключительно благодаря увеличению количества нейронов.

Для этого необходимо провести контролируемые эксперименты.

#### **3. Сравнение результатов R²**

Сопоставим твои результаты и полученную мной оценку.

##### **Эксперименты с batch_size**

| Batch size | R²           | RMSE         |
| ---------- | ------------ | ------------ |
| 64         | 0,528724     | 0,131514     |
| 128        | 0,509933     | 0,134111     |
| **256**    | **0,534347** | **0,130727** |
| 1024       | 0,527885     | 0,131631     |

Влияние batch_size на R²

Лучший результат получен при batch_size = 256.

Однако различия между 64, 256 и 1024 относительно небольшие. Особенно выделяется batch_size = 128, при котором качество ухудшается.

Почему это происходит?

Размер batch влияет на характер оптимизации:

- Маленький batch создаёт более шумные оценки градиента. Этот шум иногда помогает находить решения с лучшей обобщающей способностью.
- Большой batch обеспечивает более стабильную оценку градиента, но уменьшает количество обновлений весов за одну эпоху.
- Промежуточный размер batch часто оказывается удачным компромиссом.

В твоих экспериментах batch_size = 256 оказался наиболее удачным из проверенных вариантов.

##### **Эксперименты с количеством эпох**

| Эпохи   | R²           | RMSE         |
| ------- | ------------ | ------------ |
| 20      | 0,473984     | 0,138942     |
| 80      | 0,511734     | 0,133864     |
| **160** | **0,531746** | **0,131092** |
| 500     | 0,480007     | 0,138145     |

Зависимость R² от количества эпох

Здесь наблюдается более интересная закономерность.

**20 эпох.** Модель ещё недостаточно обучилась. Вероятно, наблюдается недообучение (underfitting).

**80 эпох.** Качество заметно улучшилось. Сеть лучше сформировала зависимости между признаками.

**160 эпох.** Получен лучший результат из исследованных вариантов. На этой стадии модель достигает наиболее удачного баланса обучения.

**500 эпох.** Качество ухудшилось. R² снизился с 0,531746 до 0,480007.

Это может свидетельствовать о переобучении (overfitting), особенно с учётом большой архитектуры и отсутствия регуляризации.

Однако важно: по одним только итоговым метрикам нельзя однозначно доказать переобучение. Для этого необходимо увидеть одновременно тренировочную и проверочную кривые ошибки. Возможна также нестабильность оптимизации из-за высокого learning rate.

#### **4. Влияние learning rate**

У тебя используется:

```
fOptimizer = tf.keras.optimizers.Adam(    learning_rate=0.01)
```

У меня:

```
optimizer = tf.keras.optimizers.Adam(    learning_rate=0.001)
```

Разница — в 10 раз.

Это существенное отличие.

Learning rate определяет величину изменения весов при обновлении градиента.

При значении 0,01:

- обучение может происходить быстрее;
- модель может быстрее достигать хорошего решения;
- возрастает риск перескакивания через минимум функции потерь;
- при большой сети возможна нестабильность оптимизации.

При значении 0,001:

- обновления весов более осторожные;
- обучение может быть медленнее;
- зачастую проще добиться устойчивого снижения ошибки.

По твоим результатам нельзя утверждать, что 0,01 — оптимальное значение. Но оно определённо заслуживает отдельного исследования.

Я бы проверил:

| Learning rate | Назначение                     |
| ------------- | ------------------------------ |
| 0,0001        | Медленное, осторожное обучение |
| 0,0005        | Промежуточный вариант          |
| 0,001         | Базовое значение               |
| 0,003         | Более быстрые обновления       |
| 0,01          | Твой исходный вариант          |

При этом важно сохранять остальные параметры неизменными.

#### **5. Сравнение результатов моей и твоей модели**

**Лучший результат твоих экспериментов**

R², batch experiment - **0,534347**

R², epoch experiment - **0,531746**

**Мой результат** - **0,4744**

MLP с архитектурой 4 → 64 → 32 → 1

Сравним лучшие значения:

| Модель              | R²           |
| ------------------- | ------------ |
| Моя MLP             | 0,4744       |
| Твоя MLP, 160 эпох  | 0,531746     |
| Твоя MLP, batch 256 | **0,534347** |

Разница между лучшим твоим результатом и моим:

$\Delta R^2=0.534347-0.4744=0.059947$

То есть примерно **5,99 процентных пункта R²**.

При условии одинаковых тестовых наблюдений это существенное улучшение.

##### **Важное замечание о RMSE**

Твой RMSE находится около 0,13, мой — около 1,04.

Их пока нельзя корректно сравнивать напрямую.

Скорее всего, использовались разные шкалы целевой переменной. Например, твой `y` мог быть нормализован, а мои метрики рассчитаны в первоначальной шкале `Median_Income`.

R² не зависит от обычного линейного масштабирования целевой переменной, поэтому его сравнение допустимо при одинаковой тестовой выборке и одинаковых целевых наблюдениях.

Для корректного сравнения RMSE необходимо:

1. Получать прогноз модели.
2. Выполнять обратное преобразование `y`, если использовался scaler.
3. Рассчитывать RMSE на исходной шкале.

Кроме того, сейчас нельзя подтвердить, что тестовые выборки в двух экспериментах полностью совпадают. Поэтому сравнение R² пока предварительное.

#### **6. Что особенно важно в твоих экспериментах**

Я бы выделил три результата.

**Первый — оптимальный batch_size не обязательно должен быть минимальным.**

У тебя 256 оказался лучше 64 и 1024. Это вполне ожидаемо: размер batch влияет на шум градиента и число обновлений параметров.

**Второй — увеличение числа эпох не гарантирует повышения точности.**

Рост качества до 160 эпох и ухудшение при 500 — важный сигнал. Для твоей большой сети особенно полезно реализовать Early Stopping.

**Третий — большая сеть не обязательно лучше маленькой.**

Несмотря на то что твоя модель показала более высокий R², она обладает примерно в 250 раз большим количеством параметров. Поэтому необходимо выяснить, насколько хорошо она обобщает закономерности на новых наблюдениях.

#### **7. Какие эксперименты я рекомендую провести дальше**

Я предлагаю не ограничиваться сравнением двух готовых моделей, а провести небольшое исследование архитектуры.

Сначала оставить твои пять признаков и параметры обучения, а менять только архитектуру.

| Эксперимент | Архитектура                   | Параметры |
| ----------- | ----------------------------- | --------- |
| A           | 5 → 64 → 32 → 1               | 2 561     |
| B           | 5 → 128 → 64 → 1              | 9 089     |
| C           | 5 → 256 → 128 → 1             | 34 817    |
| D           | 5 → 500 → 200 → 1             | 103 701   |
| E           | 5 → 1000 → 500 → 200 → 10 → 1 | 608 721   |

Затем для лучшей архитектуры проверить batch size и learning rate.

Для оценки результатов я бы использовал следующий протокол:

- Одна и та же обучающая и проверочная выборка.
- Одинаковые правила очистки выбросов.
- Одинаковое масштабирование данных.
- Фиксированный `random_seed`.
- Отдельная validation-выборка для Early Stopping и подбора параметров.
- Test-выборка используется только для окончательного сравнения.

Особенно рекомендую добавить `ReduceLROnPlateau` и `EarlyStopping`:

```
callbacks = [    tf.keras.callbacks.EarlyStopping(        monitor='val_loss',        patience=30,        restore_best_weights=True    ),    tf.keras.callbacks.ReduceLROnPlateau(        monitor='val_loss',        factor=0.5,        patience=10,        min_lr=0.00001    )]
```

Это позволит не подбирать количество эпох вручную для каждого эксперимента.

#### **8. Вывод для отчёта**

##### Сравнительный анализ многослойных персептронов

В ходе исследования были рассмотрены две архитектуры многослойного персептрона для решения задачи регрессии.

Первая архитектура содержит четыре скрытых слоя с 1000, 500, 200 и 10 нейронами соответственно. В качестве оптимизатора используется Adam с learning rate 0,01. Вторая архитектура содержит два скрытых слоя с 64 и 32 нейронами, использует Adam с learning rate 0,001, а также Dropout и L2-регуляризацию.

Первая модель обладает значительно большей вычислительной сложностью: количество обучаемых параметров составляет 608 721 против 2 433 у второй модели. Это обеспечивает более высокую гибкость при моделировании сложных нелинейных зависимостей, но одновременно увеличивает риск переобучения.

Дополнительно в первой модели используется пять входных признаков, включая Population, тогда как во второй — четыре. Таким образом, на результаты влияет не только структура сети, но и состав входных данных.

В экспериментах с размером batch наилучшее значение R² = 0,534347 получено при batch_size = 256. При увеличении batch_size до 1024 качество практически не изменилось, а при batch_size = 128 наблюдалось снижение R² до 0,509933.

В экспериментах с количеством эпох максимальное значение R² = 0,531746 достигнуто при 160 эпохах. Увеличение количества эпох до 500 привело к снижению R² до 0,480007. Такое поведение может быть связано с переобучением или нестабильностью процесса оптимизации.

Сравнение с ранее построенной компактной MLP показало преимущество более крупной сети в проведённых экспериментах: R² увеличился с 0,4744 до 0,534347. Однако для однозначного определения причин улучшения необходимо провести дополнительные эксперименты на одинаковых выборках, последовательно изменяя только один параметр модели.

Таким образом, результаты подтверждают существенное влияние архитектуры нейронной сети, состава признаков, размера пакета и количества эпох на качество регрессионного прогнозирования. В качестве дальнейшего направления исследования предлагается подбор архитектуры и гиперпараметров с применением Early Stopping и отдельной проверочной выборки.

**Моя главная рекомендация:** не отказывайся от своей большой архитектуры — она уже показала хороший результат. Но проверь, насколько этот результат устойчив. Особенно интересен эксперимент с 500 → 200 → 64 → 1: возможно, такая сеть сохранит большую часть выразительности твоей исходной архитектуры, но будет менее склонна к переобучению.

**Комментарий**

LLM пыталась объяснить результаты проведенных экспериментов. Главный вывод - сложная структура НС может объяснять более сложные зависимости, однако склонна к переобучению. К этому добавляется отсутствие колбеков на уменьшение скорости обучения и раннюю остановку. LLM также порекоммендовала экспериментировать с иными структура.

По неясной причине GPT стал формировать выводы для ранее не упомянутого отчета

### Ансамблевая модель

**ПРОМПТ**:
```
Для сравнения построй ансамблевую регрессионную модель. Обоснуй выбор типа ансамбля и его структуры.
```

**Ответ GPT**

In [ ]:
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    VotingRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

SEED = 42
TARGET = 'Median_Income'
FEATURES = [
    'Median_House_Value',
    'Distance_to_coast',
    'Tot_Rooms',
    'Median_Age',
    'Population'
]

In [ ]:
# Восстанавливаем те же train/test строки, что использовались ранее.
train_idx, test_idx = train_test_split(
    np.arange(len(df_gpt_clean)), test_size=0.2, random_state=SEED
)
train_df = df_gpt_clean.iloc[train_idx].copy()
test_df = df_gpt_clean.iloc[test_idx].copy()

X_train = train_df[FEATURES].astype(float)
y_train = train_df[TARGET].astype(float)
X_test = test_df[FEATURES].astype(float)
y_test = test_df[TARGET].astype(float)

# Validation создаётся только внутри train. Test остаётся нетронутым.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED
)

print('Исходная очищенная таблица:', df_gpt_clean.shape)
print('Обучающая выборка:', X_train.shape)
print('Внутренняя validation:', X_val.shape)
print('Проверочная выборка:', X_test.shape)


In [ ]:
# Кандидаты ансамбля с разными индуктивными предпосылками.
et = ExtraTreesRegressor(
    n_estimators=140,
    max_features=0.9,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=2
)

hgb = HistGradientBoostingRegressor(
    max_iter=160,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=SEED
)

ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=10.0))
])

# Обучаем кандидатов только на fit-части и получаем прогнозы validation.
with threadpool_limits(limits=2):
    et_val_model = et.fit(X_fit, y_fit)
    hgb_val_model = hgb.fit(X_fit, y_fit)
    ridge_val_model = ridge.fit(X_fit, y_fit)

pred_val = {
    'ExtraTreesRegressor': et_val_model.predict(X_val),
    'HistGradientBoostingRegressor': hgb_val_model.predict(X_val),
    'Ridge': ridge_val_model.predict(X_val)
}

candidate_validation = pd.DataFrame([
    {
        'Model': name,
        'Validation_MAE': mean_absolute_error(y_val, pred),
        'Validation_RMSE': mean_squared_error(y_val, pred) ** 0.5,
        'Validation_R2': r2_score(y_val, pred)
    }
    for name, pred in pred_val.items()
])
display(candidate_validation.round(5))


In [ ]:
# Подбор трёх весов на validation: веса >= 0, сумма = 1.
# Сетка 0.05 делает процедуру прозрачной и воспроизводимой.
weight_grid = np.round(np.arange(0, 1.0001, 0.05), 2)
best_mse = np.inf
best_weights = None

for w_et in weight_grid:
    for w_hgb in weight_grid:
        w_ridge = round(1.0 - w_et - w_hgb, 10)
        if w_ridge < -1e-9 or w_ridge > 1 + 1e-9:
            continue
        pred = (
            w_et * pred_val['ExtraTreesRegressor'] +
            w_hgb * pred_val['HistGradientBoostingRegressor'] +
            w_ridge * pred_val['Ridge']
        )
        mse = mean_squared_error(y_val, pred)
        if mse < best_mse:
            best_mse = mse
            best_weights = {
                'ExtraTreesRegressor': float(w_et),
                'HistGradientBoostingRegressor': float(w_hgb),
                'Ridge': float(w_ridge)
            }

print('Оптимальные веса на validation:')
print(best_weights)
print('Validation RMSE:', round(best_mse ** 0.5, 6))

# Нулевые веса исключаем из финальной структуры модели.
active_estimators = []
active_weights = []
if best_weights['ExtraTreesRegressor'] > 1e-9:
    active_estimators.append(('extra_trees', ExtraTreesRegressor(
        n_estimators=140, max_features=0.9, min_samples_leaf=2,
        random_state=SEED, n_jobs=2
    )))
    active_weights.append(best_weights['ExtraTreesRegressor'])
if best_weights['HistGradientBoostingRegressor'] > 1e-9:
    active_estimators.append(('hist_gradient_boosting', HistGradientBoostingRegressor(
        max_iter=160, learning_rate=0.05, max_leaf_nodes=15,
        min_samples_leaf=20, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=15, random_state=SEED
    )))
    active_weights.append(best_weights['HistGradientBoostingRegressor'])
if best_weights['Ridge'] > 1e-9:
    active_estimators.append(('ridge', Pipeline([
        ('scaler', StandardScaler()), ('regressor', Ridge(alpha=10.0))
    ])))
    active_weights.append(best_weights['Ridge'])

# Нормируем на случай накопления погрешности округления.
active_weights = np.asarray(active_weights, dtype=float)
active_weights = active_weights / active_weights.sum()
print('Финальные компоненты:', [name for name, _ in active_estimators])
print('Веса VotingRegressor:', active_weights)


In [ ]:
# Обучаем финальный ансамбль на всей обучающей выборке.
ensemble = VotingRegressor(
    estimators=active_estimators,
    weights=active_weights,
    n_jobs=1
)
with threadpool_limits(limits=2):
    ensemble.fit(X_train, y_train)

y_pred = ensemble.predict(X_test)

def regression_metrics(name, split, y_true, y_hat):
    mse = mean_squared_error(y_true, y_hat)
    return {
        'Model': name,
        'Split': split,
        'MAE': mean_absolute_error(y_true, y_hat),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_true, y_hat)
    }

metric_rows = []
for (name, _), fitted_model in zip(active_estimators, ensemble.estimators_):
    metric_rows.append(regression_metrics(name, 'Test', y_test, fitted_model.predict(X_test)))
metric_rows.append(regression_metrics('Weighted_VotingRegressor', 'Train', y_train, ensemble.predict(X_train)))
metric_rows.append(regression_metrics('Weighted_VotingRegressor', 'Test', y_test, y_pred))
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(6))


In [ ]:
# Диагностика предсказаний и ошибок.
errors = y_test.to_numpy() - y_pred
squared_errors = errors ** 2

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, s=12, alpha=0.4)
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
ax.plot([lo, hi], [lo, hi], '--')
ax.set_xlabel('Истинное Median_Income')
ax.set_ylabel('Предсказанное Median_Income')
ax.set_title('Ансамбль: pred vs true')
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(errors, bins=45, density=True, alpha=0.55, label='Гистограмма')
try:
    from scipy.stats import gaussian_kde
    x_grid = np.linspace(errors.min(), errors.max(), 250)
    ax.plot(x_grid, gaussian_kde(errors)(x_grid), label='KDE')
except Exception:
    pass
ax.set_xlabel('Ошибка: y_true - y_pred')
ax.set_ylabel('Плотность')
ax.set_title('Распределение ошибок ансамбля')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_pred, squared_errors, s=12, alpha=0.4)
ax.set_xlabel('Предсказанное Median_Income')
ax.set_ylabel('Квадрат ошибки')
ax.set_title('Квадрат ошибки относительно прогноза')
ax.grid(alpha=0.25)
plt.show()


In [ ]:
# График permutation importance.
plot_imp = importance_df.sort_values('Среднее снижение R2')
plt.figure(figsize=(8, 5))
plt.barh(plot_imp['Признак'], plot_imp['Среднее снижение R2'], xerr=plot_imp['Среднее снижение R2'])
plt.xlabel('Снижение R² при перемешивании признака')
plt.title('Permutation importance ансамбля')
plt.grid(axis='x', alpha=0.25)
plt.show()


Для сравнения с линейной регрессией и MLP построен **взвешенный ансамбль регрессионных моделей — Weighted Voting Regressor**. Он объединяет прогнозы нескольких алгоритмов с разными принципами обучения.

Полученный результат на проверочной выборке:

R² ансамблевой модели - **0,558501**

MAE - **0,7542**

RMSE - **0,9559**

5 признаков · 12 323 обучающих наблюдения · 3 081 проверочное наблюдение

#### **1. Обоснование выбора типа ансамбля**

Я выбрал `VotingRegressor` с взвешенным усреднением прогнозов.

Почему именно он?

В регрессионной задаче мы получаем непрерывное числовое значение. Поэтому можно объединить прогнозы нескольких моделей:

$ \hat y\_{\text{ensemble}}= \sum_{i=1}^{n}w_i\hat y_i$

При этом:

$\sum_{i=1}^{n}w_i=1,\qquad w_i\geq0$

Преимущества такого подхода:

- можно объединять модели с разными алгоритмами обучения;
- отдельные ошибки моделей частично компенсируют друг друга;
- веса позволяют учитывать качество каждого алгоритма;
- ансамбль проще интерпретировать, чем метамодель с дополнительным уровнем обучения.

В отличие от Bagging, здесь объединяются не только случайные деревья одного семейства, а алгоритмы с различными принципами построения зависимостей.

#### **2. Структура ансамбля**

Первоначально проверены три алгоритма.

**Extra Trees Regressor** - **40%**

140 деревьев · min_samples_leaf = 2

**HistGradientBoosting Regressor** - **60%**

160 итераций · learning_rate = 0,05

**Ridge Regression** - **0%**

Проверена как кандидат, но исключена из итогового ансамбля

**Weighted Voting Regressor**

##### **1 прогноз**

Взвешенное объединение двух моделей

Обоснование компонентов:

**ExtraTreesRegressor** использует множество рандомизированных деревьев. Он хорошо выявляет нелинейные зависимости и взаимодействия признаков.

**HistGradientBoostingRegressor** последовательно строит деревья, исправляя ошибки предыдущих итераций. Это позволяет выявлять остаточные закономерности, которые могут быть не обнаружены Extra Trees.

**Ridge** проверяет, насколько полезно добавить в ансамбль линейную составляющую. В результате подбора весов ему присвоено нулевое значение, поэтому в финальную модель он не вошёл.

Таким образом, финальная структура:

$ \boxed{ \hat y=0.4\hat y\_{ET}+0.6\hat y\_{HGB} }$

#### **3. Подбор весов**

Чтобы не использовать тестовую выборку при настройке ансамбля, из обучающей выборки выделена дополнительная validation-часть.

| Выборка     | Размер |
| ----------- | ------ |
| Обучающая   | 12 323 |
| Validation  | 2 465  |
| Проверочная | 3 081  |

Весовые коэффициенты подбирались перебором с шагом 0,05. Критерий выбора — минимальное значение MSE на validation.

Получены следующие результаты:

| Модель               | Validation RMSE |
| -------------------- | --------------- |
| Extra Trees          | 1,0004          |
| HistGradientBoosting | 0,9913          |
| Ridge                | 1,0732          |
| Взвешенный ансамбль  | **0,9862**      |

Ансамбль получил лучший результат на validation, поэтому выбран для окончательного тестирования.

#### **4. Результаты проверки ансамбля**

| Модель               | MAE        | RMSE       | R²         |
| -------------------- | ---------- | ---------- | ---------- |
| Extra Trees          | 0,7704     | 0,9798     | 0,5362     |
| HistGradientBoosting | 0,7544     | 0,9549     | 0,5594     |
| Ridge                | 0,8328     | 1,0413     | 0,4761     |
| **Voting Regressor** | **0,7542** | **0,9559** | **0,5585** |

На тестовой выборке ансамбль достиг R² = 0,558501.

Важное наблюдение: HistGradientBoosting отдельно показал чуть более высокий R², чем ансамбль. Разница небольшая — около 0,0009. Ансамбль немного лучше по MAE, но немного уступает по MSE и RMSE.

Следовательно, ансамблирование в данном эксперименте позволило получить устойчивый компромисс между двумя алгоритмами, но не доказало превосходство над лучшим отдельным компонентом.

#### **5. Анализ вклада признаков**

Использован метод Permutation Importance с шестью повторениями.

Открыть график важности признаков

| Признак            | Снижение R² | Относительная важность |
| ------------------ | ----------- | ---------------------- |
| Median_House_Value | 0,7042      | 53,81%                 |
| Tot_Rooms          | 0,2489      | 19,02%                 |
| Population         | 0,2132      | 16,30%                 |
| Median_Age         | 0,0917      | 7,01%                  |
| Distance_to_coast  | 0,0506      | 3,86%                  |

Главное наблюдение: `Median_House_Value` остаётся наиболее важным признаком. Однако в отличие от предыдущих моделей, ансамбль достаточно активно использует `Population` и `Tot_Rooms`.

Это подтверждает полезность пятипризнакового набора в данной задаче.

Относительную важность не следует интерпретировать как причинное влияние или строгое разложение R²: коррелирующие признаки могут разделять между собой прогнозную информацию.

#### **6. Сравнение с предыдущими моделями**

Сравнение моделей по R²

Проверочная выборка; для нового ансамбля использовано пять признаков.

По сравнению с моей предыдущей четырёхпризнаковой MLP:

$0.558501-0.474391=0.084110$

Увеличение R² составляет около 8,41 процентного пункта.

По сравнению с твоим лучшим экспериментом MLP (R² = 0,534347) разница составляет:

$0.558501-0.534347=0.024154$

То есть около 2,42 процентного пункта в пользу ансамбля. Но это сравнение предварительное: ты не указал точные индексы своей тестовой выборки, поэтому нельзя гарантировать полную сопоставимость. Твои значения RMSE около 0,13 также нельзя напрямую сравнивать с моим RMSE 0,9559 без уточнения масштаба целевой переменной.

#### **7. Диагностика прогнозов**

Дополнительно построены диаграммы:

**Истинные и предсказанные значения**

**Распределение ошибок**

**Квадрат ошибки**

У ансамбля также наблюдается разница между тренировочным R² (0,7569) и проверочным (0,5585). Это повод дополнительно исследовать обобщающую способность модели, в частности чувствительность результатов к параметрам деревьев и случайному разбиению.

#### **9. Вывод для исследовательской работы**

**Готовый текст для отчёта:**

В качестве дополнительного метода регрессионного прогнозирования была построена ансамблевая модель на основе Weighted Voting Regressor. Выбор данного метода обусловлен возможностью объединения прогнозов алгоритмов с различными принципами обучения.

В качестве базовых алгоритмов были рассмотрены ExtraTreesRegressor, HistGradientBoostingRegressor и Ridge Regression. Оптимальные веса определялись на дополнительной проверочной части обучающей выборки по критерию минимизации среднеквадратичной ошибки. В результате Ridge получил нулевой вес и был исключён из итоговой структуры. Вес ExtraTrees составил 0,4, HistGradientBoosting — 0,6.

На тестовой выборке ансамбль показал MAE = 0,7542, RMSE = 0,9559 и R² = 0,5585. Результат превосходит показатели ранее построенных линейных моделей и четырёхпризнаковой MLP. При этом отдельная модель HistGradientBoosting показала практически аналогичный результат и незначительно превзошла ансамбль по R².

Анализ перестановочной важности показал, что наибольший вклад в прогнозирование вносит Median_House_Value. Существенное значение также имеют Tot_Rooms и Population.

Таким образом, взвешенный ансамбль продемонстрировал высокую прогнозную способность и стал одним из лучших рассмотренных вариантов регрессионной модели. Однако его преимущество над отдельным алгоритмом HistGradientBoosting не является существенным, что необходимо учитывать при окончательном выборе модели.

**Моя рекомендация:** оставить три модели для итогового сравнительного исследования — твою MLP, HistGradientBoosting и Weighted Voting Regressor. Это позволит сравнить нейросетевой подход, градиентный бустинг и объединение разных алгоритмов на одной и той же выборке.

**Комментарий**

LLM был выбран VotingRegressor, но как такового сравнения с другими подходами приведено не было. GPT провела обучение на 3 отдельных моделях и определяла, какие веса им дать в ансамбле. Ансамбль получает лучшую оценку $R^2$ и лучше использует признаки кроме Median_House_Value.

**ПРОМПТ**
```
Я строил ансамбли ранее. Привел свои лучшие характеристики.

−Бэггинг  𝐵𝐵 = 1000:
o MAE: с 0.757
o MSE: 0.970
o RMSE: 0.985
o R2: 0.543
o Var: 0.397 (Дисперсия абсолютной ошибки)
- Случайный и лес с максимальной глубиной = 15, долей 𝑚𝑚/𝑝𝑝 = 0.4 и B: = 500
o MAE: 0.745
o MSE: 0.934
o RMSE: 0.966
o R2: 0.560
o Var: 0.378 (Дисперсия абсолютной ошибки)
- Градиентный бустинг с λ = 0.1 и 𝐵𝐵 = 500
o MAE: 0.749
o MSE: 0.956
o RMSE: 0.978
o R2: 0.549
o Var: 0.394 (Дисперсия абсолютной ошибки)

Проинтерпретируй результаты.
```

Твои результаты показывают, что **наилучшее качество среди трёх исследованных ансамблей обеспечивает случайный лес (Random Forest)**. Причём он превосходит Bagging и Gradient Boosting по всем пяти приведённым показателям.

Это интересный результат: дополнительная случайность при построении деревьев и ограничение их глубины в твоём эксперименте оказались эффективнее, чем простое усреднение большого количества моделей или последовательное исправление ошибок.

#### **1. Сравнение трёх ансамблевых моделей**

Сведём результаты в одну таблицу.

| Метрика | Bagging | Random Forest | Gradient Boosting |
| ------- | ------- | ------------- | ----------------- |
| MAE ↓   | 0,757   | **0,745**     | 0,749             |
| MSE ↓   | 0,970   | **0,934**     | 0,956             |
| RMSE ↓  | 0,985   | **0,966**     | 0,978             |
| R² ↑    | 0,543   | **0,560**     | 0,549             |
| Var ↓   | 0,397   | **0,378**     | 0,394             |

Стрелки показывают желательное направление изменения метрик.

**Основной результат:** Random Forest имеет максимальный R² = 0,560 и минимальные значения остальных метрик.

Это означает, что при выбранных параметрах именно случайный лес обеспечивает наилучшее качество прогнозирования среди трёх рассмотренных ансамблей.

#### **2. Интерпретация каждой модели**

##### **2.1. Bagging — усреднение большого количества моделей**

Параметр:

$B=1000$

Bagging строит большое количество базовых моделей на различных bootstrap-выборках и усредняет их прогнозы.

Получены следующие характеристики:

- MAE = 0,757;
- RMSE = 0,985;
- R² = 0,543;
- Var = 0,397.

Bagging обеспечивает достаточно хорошее качество прогнозирования.

Его главное достоинство — снижение дисперсии модели за счёт усреднения большого количества независимых или слабо коррелированных прогнозов.

Однако увеличение количества моделей до 1000 не гарантирует максимального качества.

Почему?

Если базовые модели имеют похожую структуру и используют практически одинаковые правила построения дерева, их ошибки могут быть сильно коррелированы. В таком случае усреднение не обеспечивает настолько существенного улучшения, насколько можно было бы ожидать.

Также после определённого количества базовых моделей добавление новых деревьев даёт всё меньший прирост качества.

**Вывод:** Bagging обеспечивает хорошее снижение дисперсии, однако в твоём эксперименте простого усреднения недостаточно для достижения максимального качества.

##### **2.2. Random Forest — лучший результат**

Параметры:

- максимальная глубина дерева — 15;
- доля признаков $(m/p=0,4)$;
- количество деревьев $(B=500)$.

Получены:

$\boxed{R^2=0,560}$

Это лучший результат среди трёх моделей.

Особенно интересен выбор доли признаков $(m/p=0,4)$.

В твоём наборе используется пять признаков. Если 40% признаков случайно выбираются в качестве кандидатов при каждом разбиении, то:

$m=5\cdot0,4=2$

То есть при выборе разбиения дерева рассматриваются два случайных признака.

Это важное отличие Random Forest от обычного Bagging.

В Bagging деревья могут использовать весь набор признаков. В Random Forest дополнительно применяется случайный отбор признаков, благодаря чему деревья становятся менее похожими друг на друга.

Это позволяет уменьшать корреляцию между отдельными деревьями.

$(Var(X) = E(X^2) - [E(X)]^2)$

**Почему это важно?**

При усреднении прогнозов уменьшается влияние случайных ошибок отдельных деревьев. Но эффективность такого усреднения зависит от того, насколько эти ошибки коррелированы.

Случайный отбор признаков помогает уменьшать эту зависимость.

Второй важный параметр — максимальная глубина 15.

Ограничение глубины:

- препятствует чрезмерному усложнению деревьев;
- уменьшает риск запоминания отдельных наблюдений;
- помогает контролировать компромисс между смещением и дисперсией.

В результате комбинация случайного отбора признаков, ограничения глубины и усреднения 500 деревьев дала наилучший результат.

**Вывод:** Random Forest наиболее удачно сбалансировал сложность базовых моделей и их разнообразие.

##### **2.3. Gradient Boosting — последовательное исправление ошибок**

Параметры:

$\lambda=0,1,\qquad B=500$

Здесь $\lambda$ — скорость обучения (learning rate), а $B$ — количество последовательных базовых моделей.

Gradient Boosting работает иначе, чем Bagging и Random Forest.

Если Bagging и Random Forest строят ансамбль деревьев независимо, то Gradient Boosting обучает деревья последовательно.

Каждое последующее дерево пытается исправить ошибки предыдущего ансамбля.

Механизм можно представить так:

**Первое дерево**

Начальный прогноз

**Анализ ошибок**

Определение остаточных отклонений

**Следующее дерево**

Исправление части ошибок

**Повторение 500 раз**

Постепенное улучшение ансамбля

**Итоговый прогноз**

Небольшое значение learning rate = 0,1 означает, что вклад каждой новой модели ограничен. Это помогает делать обучение более постепенным.

Количество деревьев 500 компенсирует более медленное обучение.

Результаты:

- MAE = 0,749;
- RMSE = 0,978;
- R² = 0,549;
- Var = 0,394.

Gradient Boosting занял второе место по R².

Он превосходит Bagging по всем пяти показателям, но немного уступает Random Forest.

**Вывод:** последовательное исправление ошибок оказалось эффективнее обычного усреднения, но не обеспечило такого результата, как случайный лес с выбранными параметрами.

#### **3. Почему Random Forest оказался лучше?**

Главная причина, на мой взгляд, — удачная комбинация трёх механизмов:

1. Bootstrap-формирование обучающих выборок.
2. Случайный отбор признаков при построении деревьев.
3. Ограничение глубины дерева до 15.

Особенно важен второй механизм.

В наборе присутствуют признаки, которые могут содержать пересекающуюся информацию. Например, `Tot_Rooms` и `Population` могут быть связаны с характеристиками жилого района.

При использовании всех признаков деревья могут многократно выбирать одни и те же наиболее информативные признаки.

Случайное ограничение набора кандидатов заставляет деревья находить альтернативные закономерности.

Это способствует уменьшению корреляции прогнозов отдельных деревьев.

В твоём случае такой подход оказался наиболее эффективным.

Однако нельзя утверждать, что глубина 15 и доля 0,4 являются глобально оптимальными. Это лучшие параметры среди исследованных вариантов, а не доказанный глобальный максимум.

#### **4. Что показывает Var — дисперсия абсолютной ошибки?**

Ты дополнительно рассчитывал дисперсию абсолютной ошибки:

$Var(|e|)$

Это полезное дополнение к стандартным метрикам.

В отличие от MAE, которая показывает среднюю величину абсолютной ошибки, дисперсия характеризует разброс этих ошибок относительно их среднего значения.

| Модель            | Var абсолютной ошибки |
| ----------------- | --------------------- |
| Bagging           | 0,397                 |
| Random Forest     | **0,378**             |
| Gradient Boosting | 0,394                 |

Random Forest имеет наименьшее значение.

Это означает, что абсолютные ошибки его прогнозов имеют наименьший разброс среди трёх рассмотренных моделей.

Иными словами, Random Forest не только обеспечивает меньшую среднюю ошибку, но и более стабильное распределение её величины.

При этом дисперсию нельзя рассматривать отдельно от MAE: модель теоретически может иметь небольшой разброс ошибок, но систематически ошибаться на большую величину.

В твоём случае Random Forest лучше сразу по обоим показателям.

#### **5. Сравнение улучшения относительно Bagging**

Возьмём Bagging в качестве базового варианта.

| Показатель | Bagging | Random Forest | Улучшение |
| ---------- | ------- | ------------- | --------- |
| MAE        | 0,757   | 0,745         | −1,58%    |
| MSE        | 0,970   | 0,934         | −3,71%    |
| RMSE       | 0,985   | 0,966         | −1,93%    |
| R²         | 0,543   | 0,560         | +1,7 п.п. |
| Var        | 0,397   | 0,378         | −4,79%    |

Особенно интересно снижение MSE на 3,71%.

Это говорит о том, что Random Forest лучше справляется не только с обычными ошибками, но и в определённой степени уменьшает влияние крупных отклонений прогнозов.

#### **6. Сопоставление с ранее построенным мной ансамблем**

Ранее мы построили Weighted Voting Regressor из Extra Trees и HistGradientBoosting.

Его результаты:

| Метрика | Твой Random Forest | Мой Voting |
| ------- | ------------------ | ---------- |
| MAE     | **0,745**          | 0,7542     |
| RMSE    | 0,966              | **0,9559** |
| R²      | **0,560**          | 0,5585     |
| MSE     | 0,934              | ≈0,9137    |

Здесь есть интересный момент: Random Forest немного выигрывает по R² и MAE, но уступает по RMSE и MSE.

Однако эти результаты нельзя считать прямым доказательством превосходства одной модели над другой.

При одинаковой тестовой выборке и одной шкале целевой переменной более высокий R² обязательно соответствует меньшему MSE и RMSE. Здесь наблюдается обратное соотношение.

Следовательно, перед итоговым сравнением необходимо проверить, совпадают ли:

- индексы тестовых наблюдений;
- масштаб целевой переменной;
- правила обработки выбросов;
- способ расчёта метрик.

Для окончательного исследования я бы обязательно провёл оценку всех моделей на одном и том же тестовом наборе.

#### **7. Общий рейтинг рассмотренных ансамблей**

1 место

**Random Forest** - **0,560**

Лучшее сочетание R², MAE, MSE, RMSE и дисперсии абсолютной ошибки.

2 место

**Gradient Boosting** - **0,549**

Хорошее качество за счёт последовательного исправления ошибок.

3 место

**Bagging** - **0,543**

Надёжное базовое ансамблирование, но в данном эксперименте менее эффективное.

#### **8. Вывод для исследовательской работы**

Ниже текст, который можно использовать в отчёте.

##### Сравнительный анализ ансамблевых регрессионных моделей

В ходе исследования были рассмотрены три ансамблевых метода регрессии: Bagging, Random Forest и Gradient Boosting.

Для алгоритма Bagging использовано 1000 базовых моделей. Получены MAE = 0,757, MSE = 0,970, RMSE = 0,985 и R² = 0,543. Метод обеспечивает снижение дисперсии прогнозирования за счёт усреднения результатов моделей, обученных на различных bootstrap-выборках.

Для Random Forest использовано 500 деревьев с максимальной глубиной 15 и долей случайно выбираемых признаков m/p = 0,4. Получены MAE = 0,745, MSE = 0,934, RMSE = 0,966 и R² = 0,560. Данная модель показала лучшие результаты по всем рассмотренным метрикам.

Преимущество Random Forest можно объяснить дополнительной рандомизацией признаков при построении деревьев. В отличие от обычного Bagging, данный алгоритм не только использует bootstrap-выборки, но и ограничивает набор признаков, рассматриваемых при разбиении узлов. Это способствует уменьшению корреляции между отдельными деревьями и повышению эффективности усреднения их прогнозов. Ограничение максимальной глубины также позволяет контролировать сложность базовых моделей.

Для Gradient Boosting использовано 500 последовательных базовых моделей с коэффициентом обучения λ = 0,1. Получены MAE = 0,749, MSE = 0,956, RMSE = 0,978 и R² = 0,549. Метод показал более высокое качество по сравнению с Bagging, что свидетельствует об эффективности последовательного исправления ошибок предыдущих моделей. Однако по всем метрикам он незначительно уступил Random Forest.

Дополнительно рассчитана дисперсия абсолютной ошибки. Наименьшее значение получено для Random Forest — 0,378, что свидетельствует о меньшем разбросе абсолютных ошибок прогнозирования.

Таким образом, среди трёх рассмотренных ансамблевых методов наилучшие результаты продемонстрировал Random Forest. Его использование случайного отбора признаков, ограничения глубины и усреднения 500 деревьев обеспечило наиболее удачное сочетание точности и стабильности прогнозирования.

В качестве наиболее перспективной ансамблевой модели для дальнейшего исследования рекомендуется Random Forest с максимальной глубиной 15, долей признаков m/p = 0,4 и количеством деревьев B = 500.

#### **9. Моя рекомендация по следующему этапу**

Я бы не ограничивался уже полученными результатами. У тебя есть хорошая экспериментальная база, и теперь полезнее всего провести единое сравнительное исследование:

- твоя MLP с пятью признаками;
- Random Forest с параметрами 15, 0,4 и 500;
- Gradient Boosting с параметрами 0,1 и 500;
- наш Weighted Voting Regressor.

Особенно рекомендую дополнительно исследовать Random Forest при `max_depth` 10, 15, 20 и `max_features` 0,2, 0,4, 0,6, 0,8.

Это позволит проверить, насколько результат зависит от глубины деревьев и степени случайного отбора признаков, а также установить, действительно ли найденная конфигурация близка к оптимальной.

**Комментарий**

Ручной RandomForest оказался наилучшей среди построенных моделью. LLM попыталась объяснить, почему наблюдается такой результат, но также указала, что это не доказанный глобальный максимум, а лишь локальный подобранный экспериментально.

## DeepSeek-V4.1-Flash-1017

### Анализ и подготовка датасета

**ПРОМПТ**:

```
Проведи предварительный анализ приложенного датасета. Определи:
- Символ, используемый для разделения ячеек
- Символ, используемый разделения дробной и целой части
- Наличие нечисловых данных
- Наличие штампов даты или времени
- Наличие заголовка

Создай Dataframe `df_deepseek`, получи размеры исходной таблицы. Определи наличие пропусков и выбери способ их обработки. Получи размер таблицы после обработки пропусков.
Определи наличие и приведи нужные для работы признаки к числовому типу (цель работы - построение регрессионных моделей).
Сохрани в виде файла `dataset_prepared_deepseek.csv` текущее состояние обработанной таблицы.

Работа ведется в Jupyter notebook - ответы представляй блоками по смыслу.

Библиотеки в дальнейшем: pandas, numpy, matplotlib, scikit-learn, statsmodels, TensorFlow/Keras
```

* Приложены оригинальные csv и пояснение к нему
* Включен режим *DeepThink*

**Ответ Deepseek**

1. Предварительный анализ
    - Разделитель ячеек: ;
    - Разделитель целой и дробной части: . — точка
    - Нечисловые данные:
        - признак ocean_proximity — категориальный;
        - в Median_Income встречается маркер пропуска ?$?.
    - Штампы даты/времени: отсутствуют.
    - Заголовок: присутствует в первой строке.
    - Целевая переменная для регрессии: median_house_value.
    - В таблице есть дублирующийся по смыслу столбец Median_House_Value, значения которого совпадают с median_house_value. Для регрессии его лучше удалить, чтобы не было утечки целевой переменной.

2. Чтение датасета и размер исходной таблицы

In [ ]:
import numpy as np
import pandas as pd

df_deepseek = pd.read_csv(
    "California_Houses_15_col_dot.csv",
    sep=";",
    decimal=".",
    na_values=["?$?"]
)

print(df_deepseek.shape)

Ожидаемый результат:
```
(20640, 16)
```
То есть исходно: 20 640 строк × 16 столбцов.

3. Проверка пропусков и их обработка

In [ ]:
print(df_deepseek.isna().sum())

Ожидаемо пропуск только в Median_Income:
```
Median_Income    1
dtype: int64
```
Так как пропуск всего один, выбираем удаление строки — потеря менее 0.01% данных и не требует искусственного заполнения.

In [ ]:
# до исправления
# df_deepseek = df_deepseek.dropna(subset=["Median_Income"]).reset_index(drop=True)

# после
df_deepseek = df_deepseek.dropna().reset_index(drop=True)
print(df_deepseek.shape)

Результат после обработки пропусков:
```
(20639, 16)
```

4. Приведение признаков к числовому типу

In [ ]:
# Удаляем дублирующий целевой признак
df_deepseek = df_deepseek.drop(columns=["Median_House_Value"])

# Все столбцы, кроме категориального ocean_proximity, приводим к числу
numeric_cols = df_deepseek.columns.drop("ocean_proximity")
df_deepseek[numeric_cols] = df_deepseek[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Проверка, что после замены ?$? пропусков не осталось
print(df_deepseek[numeric_cols].isna().sum().sum())

Ожидаемо:
```
0
```
Категориальный признак ocean_proximity кодируем one-hot-кодированием. Для регрессии это удобнее, чем LabelEncoder.

In [ ]:
df_deepseek = pd.get_dummies(
    df_deepseek,
    columns=["ocean_proximity"],
    drop_first=True,
    dtype=int
)

print(df_deepseek.shape)
print(df_deepseek.dtypes)

Итоговый ожидаемый размер:
```
(20639, 18)
```

5. Сохранение подготовленного датасета

In [ ]:
df_deepseek.to_csv(
    "dataset_prepared_deepseek.csv",
    index=False,
    sep=";",
    decimal="."
)

Файл dataset_prepared_deepseek.csv сохранён в подготовленном виде: пропуски обработаны, категориальный признак закодирован, все признаки приведены к числовому типу.

**Комментарий**

В отличие от GPT 5.6 модель Deepseek не выполняет действительных вычислений / не исполняет код, а делает предположения по тексту. Модель заметила нулевое значение `?$?`, но проигнорировала пропуски - из-за чего возникли ошибки, которые модель не смогла обнаружить; причина описана выше.

Ручные исправления - дроп **всех** пропусков, а не только по `Median_Income`

### Корреляционный анализ

**ПРОМПТ**:

```
Нужно провести корреляционный анализ всех признаков и составить матрицу корреляции.
Код должен проверить, есть ли выбросы в данных и исключить их. Построй гистрограммы распределений значений и boxplot до и после исключения выбросов. Границы диапазонов для исключения выбросов должны определяться в коде, предложи критерий.
```
* Включен режим *DeepThink*

**Ответ Deepseek**

Блок 1. Загрузка данных и корреляционный анализ

Загружаем подготовленный на предыдущем шаге датасет. Все признаки (включая one-hot-кодированный ocean_proximity) уже числовые, поэтому корреляционная матрица строится по всем 18 столбцам. Дополнительно к Пирсону считаем Спирмена для контроля на нелинейность/скошенность.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df_deepseek = pd.read_csv("dataset_prepared_deepseek.csv", sep=";", decimal=".")

numeric_cols = df_deepseek.select_dtypes(include=[np.number]).columns.tolist()
target = "Median_Income"

print("Размер таблицы:", df_deepseek.shape)
print("Числовые признаки:", numeric_cols)

# Матрицы корреляции
corr_pearson  = df_deepseek[numeric_cols].corr(method="pearson")
corr_spearman = df_deepseek[numeric_cols].corr(method="spearman")

# Тепловая карта Пирсона
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(corr_pearson.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(numeric_cols)))
ax.set_yticklabels(numeric_cols, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Матрица корреляции Пирсона (все признаки)")
plt.tight_layout()
plt.show()

# Тепловая карта Спирмена
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(corr_spearman.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(numeric_cols)))
ax.set_yticklabels(numeric_cols, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Матрица корреляции Спирмена (все признаки)")
plt.tight_layout()
plt.show()

# Топ-корреляции с целевой переменной
print("\nКорреляция с Median_Income (Пирсон):")
print(corr_pearson[target].drop(target).sort_values(key=abs, ascending=False))

print("\nКорреляция с Median_Income (Спирмен):")
print(corr_spearman[target].drop(target).sort_values(key=abs, ascending=False))

Ожидаемые ключевые выводы:

- Самая сильная положительная связь с целевой переменной — Median_Income (≈ 0.69 по Пирсону) и Median_House_Value мы уже удалили.

- Distance_to_coast, Distance_to_LA и координаты (Latitude, Longitude) — сильные отрицательные/положительные связи.

- One-hot-признаки ocean_proximity_* показывают умеренные корреляции с целевой.

- Различия между Пирсоном и Спирменом указывают на нелинейные/скошенные зависимости (в первую очередь у Population, Tot_Rooms, дистанций).

Блок 2. Поиск мультиколлинеарности

In [ ]:
high_corr = []
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        c = corr_pearson.iloc[i, j]
        if abs(c) > 0.8:
            high_corr.append((numeric_cols[i], numeric_cols[j], round(c, 3)))

print("Пары признаков с |r| > 0.8:")
for a, b, c in high_corr:
    print(f"  {a:35s} — {b:35s}: {c}")

Ожидаемый результат: пары Tot_Rooms ↔ Households (~0.98), Tot_Bedrooms ↔ Tot_Rooms (~0.93), Households ↔ Population (~0.91), а также группы Distance_to_* между собой и Distance_to_coast с координатами. Это важно для линейных моделей и statsmodels (VIF) — либо оставлять один из пары, либо использовать регуляризацию.

Блок 3. Критерий выбросов и их удаление

Критерий (Tukey's IQR fence):
Для каждого непрерывного признака вычисляются квартили Q1,Q3Q1​,Q3​ и IQR=Q3−Q1IQR=Q3​−Q1​. Значения вне интервала [Q1−k⋅IQR,Q3+k⋅IQR][Q1​−k⋅IQR,Q3​+k⋅IQR] считаются выбросами.

Множитель k выбираем адаптивно:

- k=1.5k=1.5 — для median_house_value и Median_Income. Это стандартный «умеренный» порог, который также автоматически отсекает известные cap-значения датасета (500001 для целевой и 15.0001 для дохода) — они искажают линейную регрессию.

- k=3.0k=3.0 — для остальных непрерывных признаков. Признаки Population, Tot_Rooms, Households, Tot_Bedrooms и все Distance_to_* имеют тяжёлые правые хвосты; при k=1.5k=1.5 удалялось бы 15–25 % строк, что потеря данных. Порог k=3.0k=3.0 оставляет «нормальные» большие значения и удаляет только экстремальные выбросы.

- One-hot-признаки (0/1) выбросов не имеют — не анализируются.

Строка удаляется, если она помечена как выброс хотя бы по одному признаку.

In [ ]:
# Критерий: IQR-фенсы
k_map = {}
for c in numeric_cols:
    if df_deepseek[c].nunique() <= 2:      # one-hot / бинарные — пропускаем
        continue
    k_map[c] = 1.5 if c in (target, "median_house_value") else 3.0

def iqr_mask(df, k_map):
    mask = pd.Series(False, index=df.index)
    bounds = {}
    for c, k in k_map.items():
        q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - k * iqr, q3 + k * iqr
        bounds[c] = (lo, hi)
        mask |= (df[c] < lo) | (df[c] > hi)
    return mask, bounds

mask_out, bounds = iqr_mask(df_deepseek, k_map)
print(f"Всего строк:                 {len(df_deepseek)}")
print(f"Обнаружено строк-выбросов:   {int(mask_out.sum())} ({mask_out.mean()*100:.2f} %)")

# до
#df_deepseek = df_deepseek[~mask_out].reset_index(drop=True)
# после
df_deepseek_before = df_deepseek.copy()
df_deepseek = df_deepseek[~mask_out].reset_index(drop=True)
print(f"Размер после удаления:       {df_deepseek.shape}")

Ожидаемо удаляется порядка 1 500 – 3 000 строк (≈ 7–15 %). Если после запуска удаление превысит ~20 %, критерий можно ослабить: поднять k для тяжёлохвостых признаков до 5.0 либо применить np.log1p перед IQR — логика в коде меняется в одном месте (k_map).

Блок 4. Гистограммы распределений до и после удаления выбросов

In [ ]:
n = len(numeric_cols)
ncols, nrows = 5, int(np.ceil(n / 5))

def plot_hists(df, title, color):
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 3.6 * nrows))
    axes = axes.flatten()
    for i, c in enumerate(numeric_cols):
        axes[i].hist(df[c].dropna(), bins=50, color=color, edgecolor="none")
        axes[i].set_title(c, fontsize=9)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.suptitle(title, y=1.01, fontsize=12)
    plt.tight_layout()
    plt.show()

plot_hists(df_deepseek_before, "Гистограммы ДО удаления выбросов", "steelblue")
plot_hists(df_deepseek, "Гистограммы ПОСЛЕ удаления выбросов", "seagreen")

Блок 5. Boxplot до и после удаления выбросов

In [ ]:
def plot_boxplots(df, title, color):
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 3.0 * nrows))
    axes = axes.flatten()
    for i, c in enumerate(numeric_cols):
        axes[i].boxplot(df[c].dropna(), vert=False, patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.6),
                        flierprops=dict(marker=".", markersize=2))
        axes[i].set_title(c, fontsize=9)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.suptitle(title, y=1.01, fontsize=12)
    plt.tight_layout()
    plt.show()

# «ДО» строится на исходной таблице до фильтрации — сохраняем её копию
# (если блок 3 уже выполнен, восстановим из исходного CSV)
# до
#df_before = pd.read_csv("dataset_prepared_deepseek.csv", sep=";", decimal=".")
# его же использует далее

plot_boxplots(df_deepseek_before,   "Boxplot ДО удаления выбросов",  "steelblue")
plot_boxplots(df_deepseek, "Boxplot ПОСЛЕ удаления выбросов", "seagreen")

Что увидим на графиках:

- До фильтрации у Population, Tot_Rooms, Tot_Bedrooms, Households и Distance_to_* — длинные хвосты и множество точек за усами.
- У median_house_value — характерный «шип» на 500001 (cap). После удаления он исчезает, распределение становится плавным.
- После фильтрации по IQR-фенсам большинство распределений становятся компактнее; тяжёлые хвосты убираются частично (порог k=3.0), а целевая и Median_Income — полностью «вычищены».

Блок 6. Сохранение очищенного датасета

In [ ]:
df_deepseek.to_csv(
    "dataset_no_outliers_deepseek.csv",
    index=False, sep=";", decimal="."
)
print("Сохранено:", df_deepseek.shape)

В df_deepseek теперь лежит очищенная от выбросов таблица, готовая к построению регрессионных моделей (pandas / sklearn / statsmodels / Keras). Для линейных моделей и statsmodels рекомендуется удалить один из пары сильно коррелирующих признаков (Tot_Rooms ↔ Households, Tot_Bedrooms ↔ Tot_Rooms, группу Distance_to_*), чтобы избежать мультиколлинеарности; для деревьев и нейросетей оставить можно.

**Комментарий**

Дополнительно к коэффициенту Пирсона был добавлен к. Спирмена. Также проведен анализ зависимости признаков между собой. Не ясно, откуда Deepseek берет свои предположения. Выбросы очищены подобным GPT 5.6 методом, но с адаптивным коэффициентом.

Проблемы - 2 одинаковые гистограммы - следствие одних данных, ненужное чтение исходного файла датасета в следующей диаграмме. LLM проигнорировала, что цель - Median_Income, и выбрала median_house_value

Исправления - сохранение копии изначального датасета до очистки, переопределена цель

### Выбор признаков для регрессии

**ПРОМПТ**

```
Получены следующие зависимости:
Корреляция с Median_Income (Пирсон):
median_house_value            0.688059
Distance_to_coast            -0.243421
ocean_proximity_INLAND       -0.237485
Tot_Rooms                     0.198079
Median_Age                   -0.119126
Latitude                     -0.079864
Distance_to_LA               -0.065484
ocean_proximity_NEAR BAY      0.056076
Distance_to_SanDiego         -0.055313
Distance_to_SanJose          -0.036742
ocean_proximity_NEAR OCEAN    0.027356
Distance_to_SanFrancisco     -0.022359
Longitude                    -0.015113
Households                    0.013059
ocean_proximity_ISLAND       -0.009228
Tot_Bedrooms                 -0.008070
Population                    0.004866
Name: Median_Income, dtype: float64

Корреляция с Median_Income (Спирмен):
median_house_value            0.676756
Tot_Rooms                     0.271418
ocean_proximity_INLAND       -0.254775
Distance_to_coast            -0.221398
Median_Age                   -0.147429
Latitude                     -0.088102
Distance_to_SanDiego         -0.072950
ocean_proximity_NEAR BAY      0.059941
Distance_to_SanJose          -0.045859
Distance_to_LA               -0.033608
Households                    0.030389
Distance_to_SanFrancisco     -0.026634
ocean_proximity_NEAR OCEAN    0.024664
ocean_proximity_ISLAND       -0.010777
Longitude                    -0.009851
Population                    0.006354
Tot_Bedrooms                 -0.004988
Name: Median_Income, dtype: float64

Пары признаков с |r| > 0.8:
  Tot_Rooms                           — Tot_Bedrooms                       : 0.93
  Tot_Rooms                           — Population                         : 0.857
  Tot_Rooms                           — Households                         : 0.918
  Tot_Bedrooms                        — Population                         : 0.878
  Tot_Bedrooms                        — Households                         : 0.98
  Population                          — Households                         : 0.907
  Latitude                            — Longitude                          : -0.925
  Latitude                            — Distance_to_LA                     : 0.942
  Latitude                            — Distance_to_SanDiego               : 0.992
  Latitude                            — Distance_to_SanJose                : -0.855
  Latitude                            — Distance_to_SanFrancisco           : -0.897
  Longitude                           — Distance_to_LA                     : -0.892
  Longitude                           — Distance_to_SanDiego               : -0.958
  Longitude                           — Distance_to_SanJose                : 0.924
  Longitude                           — Distance_to_SanFrancisco           : 0.955
  Distance_to_LA                      — Distance_to_SanDiego               : 0.951
  Distance_to_LA                      — Distance_to_SanFrancisco           : -0.849
  Distance_to_SanDiego                — Distance_to_SanJose                : -0.888
  Distance_to_SanDiego                — Distance_to_SanFrancisco           : -0.928
  Distance_to_SanJose                 — Distance_to_SanFrancisco           : 0.99
Выбери независимые признаки для построения регрессионных моделей: 1 и несколько. Раздели очищенный датасет на обучающую и тестовую выборки для случаев одного и нескольких признаков по-отдельности.

Исправления - таргет Median_Income. Правки в код внес самостоятельно
```

* DeepThink

**Ответ DeepSeek**

Блок 1. Обоснование выбора признаков

Целевая переменная теперь — Median_Income. По матрице корреляций и списку пар с |r| > 0.8 выделяются две явные группы коллинеарных признаков, из которых нужно оставить по одному представителю:

|Группа|	Признаки в группе|	Лучший по корреляции с целью|
|-|-|-|
|«Домохозяйства»|	Tot_Rooms, Tot_Bedrooms, Population, Households	| Tot_Rooms (0.198 Пирсон / 0.271 Спирмен) |
|«География»|	Latitude, Longitude, Distance_to_LA, Distance_to_SanDiego, Distance_to_SanJose, Distance_to_SanFrancisco, Distance_to_coast | Distance_to_coast (-0.243 / -0.221) |

Независимые от этих групп и значимые для цели:
- median_house_value — сильнейший предиктор (0.688 / 0.677);
- Median_Age (-0.119 / -0.147);
- ocean_proximity_INLAND (-0.237 / -0.255) — не входит в список |r| > 0.8 ни с одним другим признаком, значит, дополнительной информации о географии не дублирует.

Итоговый набор признаков для мульти-модели:
median_house_value, Distance_to_coast, Tot_Rooms, Median_Age, ocean_proximity_INLAND.

Блок 2. Разбиение для случая одного признака

Один признак — самый сильный предиктор median_house_value.

In [ ]:
from sklearn.model_selection import train_test_split

TARGET = "Median_Income"
RANDOM_STATE = 42
TEST_SIZE = 0.2

y = df_deepseek[TARGET]

# --- Один признак ---
features_single = ["median_house_value"]
X_single = df_deepseek[features_single]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_single, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Single feature:", features_single)
print("train:", X_train_s.shape, " test:", X_test_s.shape)

Блок 3. Разбиение для случая нескольких признаков

Набор без мультиколлинеарности (по одному представителю от каждой коллинеарной группы + независимые).

In [ ]:
features_multi = [
    "median_house_value",
    "Distance_to_coast",
    "Tot_Rooms",
    "Median_Age",
    "ocean_proximity_INLAND",
]

X_multi = df_deepseek[features_multi]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Multi features:", features_multi)
print("train:", X_train_m.shape, " test:", X_test_m.shape)

Блок 4. Контроль: проверка корреляций внутри выбранного набора

Убеждаемся, что среди выбранных признаков нет пар с |r| > 0.8.

In [ ]:
import pandas as pd

chk_single = df_deepseek[features_single + [TARGET]].corr().loc[features_single, [TARGET]]
chk_multi  = df_deepseek[features_multi  + [TARGET]].corr()

print("Корреляция с целью (single):")
print(chk_single)

print("\nМатрица корреляции выбранных признаков (multi):")
print(chk_multi.round(3))

# Проверка максимальной межпризнаковой корреляции
feat_corr_m = chk_multi.loc[features_multi, features_multi].abs()
# до
# np.fill_diagonal(feat_corr_m.values, 0)
# после
values_copy = feat_corr_m.values.copy()
np.fill_diagonal(values_copy, 0)

print("\nМаксимальная |r| между признаками multi:",
      feat_corr_m.values.max().round(3))

Ожидаемо: максимальная межпризнаковая корреляция в features_multi будет < 0.8 — набор пригоден для линейных моделей (OLS, Ridge/Lasso, statsmodels без высокого VIF).

Блок 5. Сохранение индексов и выборок (для дальнейших моделей)

Чтобы разбиения для single- и multi-моделей были согласованы (одинаковые объекты в train/test, только разный набор X), используем общий train_test_split от индекса и потом нарезаем.

In [ ]:
idx_all = df_deepseek.index.to_numpy()
idx_train, idx_test = train_test_split(
    idx_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_train_s, X_test_s = df_deepseek.loc[idx_train, features_single], df_deepseek.loc[idx_test, features_single]
X_train_m, X_test_m = df_deepseek.loc[idx_train, features_multi],  df_deepseek.loc[idx_test, features_multi]
y_train, y_test     = df_deepseek.loc[idx_train, TARGET],          df_deepseek.loc[idx_test, TARGET]

print("train:", len(idx_train), " test:", len(idx_test))
print("single:", X_train_s.shape, X_test_s.shape)
print("multi :", X_train_m.shape, X_test_m.shape)

Теперь X_train_s / X_test_s / X_train_m / X_test_m / y_train / y_test готовы к подаче в любую регрессионную модель (sklearn, statsmodels, Keras). Для Keras и statsmodels дополнительно понадобится масштабирование (StandardScaler) — оно выполняется в следующих блоках.

**Комментарий**

Для многокритериальной регрессии выбраны 5 признаков, при этом один из них - закодированное расстояние до побережья, а другой расстояние до побережья в метрах. Проверка |r| учла корреляцию признака с самим собой - проигнорировано.

### Построение линейной регрессионной модели от ОДНОГО признака

**ПРОМПТ**:

```
Построй и проанализируй линейную регрессионную модель (tf/keras) от одного признака.
Для анализа построй график разброса тренировочных и проверочных данных (целевой y и независимый признак x):
1. Отложи на нем полученную линию регрессии. Выведи формулу y=f(x) с полученными весами.
2. Построй плотность распределения и гистограмму ошибок. 
3. Построй диаграммы рассеяния вычисленных значений (pred/true) и диаграмму рассеяния квадрата ошибки.
4. Получи и выведи метрики MAE/MSE/RMSE/R2

Вынеси пункты 2-4 в отдельные функции для дальнейшего использования для других моделей.

Для метрик используй scikit-learn.
```

* Включен редим DeepThink

**Ответ DeepSeek**

Блок 1. Вспомогательные функции (метрики + диагностика)

Вынесем метрики и графики 2–4 в отдельные функции — далее они переиспользуются для всех последующих моделей.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.nonparametric.kde import KDEUnivariate

TARGET = "Median_Income"

# ---------- Метрики ----------
def compute_metrics(y_true, y_pred):
    """MAE / MSE / RMSE / R2."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "MSE":  mse,
        "RMSE": np.sqrt(mse),
        "R2":   r2_score(y_true, y_pred),
    }

def print_metrics(name, metrics):
    print(f"[{name}]")
    for k, v in metrics.items():
        print(f"  {k:5s}: {v:.6f}")

# ---------- Графики ----------
def plot_error_distribution(y_true, y_pred, title="Error distribution", ax=None):
    err = np.asarray(y_true).ravel() - np.asarray(y_pred).ravel()
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    ax.hist(err, bins=60, density=True, alpha=0.6,
            color="steelblue", edgecolor="none", label="Histogram")
    kde = KDEUnivariate(err); kde.fit()
    ax.plot(kde.support, kde.density, color="crimson", lw=2, label="KDE")
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_xlabel("error = y_true - y_pred"); ax.set_ylabel("density")
    ax.set_title(title); ax.legend()
    return ax

def plot_pred_true(y_true, y_pred, title="Pred vs True", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    y_true = np.asarray(y_true).ravel(); y_pred = np.asarray(y_pred).ravel()
    ax.scatter(y_true, y_pred, s=8, alpha=0.4, color="steelblue")
    lo = min(y_true.min(), y_pred.min()); hi = max(y_true.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="y = x")
    ax.set_xlabel("true"); ax.set_ylabel("predicted")
    ax.set_title(title); ax.legend()
    return ax

def plot_squared_error(y_true, y_pred, title="Squared error vs True", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    err2 = (np.asarray(y_true).ravel() - np.asarray(y_pred).ravel()) ** 2
    ax.scatter(y_true, err2, s=8, alpha=0.4, color="darkorange")
    ax.set_xlabel("true"); ax.set_ylabel("squared error")
    ax.set_title(title)
    return ax

def plot_diagnostics(y_true, y_pred, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    plot_error_distribution(y_true, y_pred, title=f"{title} — error distribution", ax=axes[0])
    plot_pred_true(y_true, y_pred, title=f"{title} — pred vs true", ax=axes[1])
    plot_squared_error(y_true, y_pred, title=f"{title} — squared error", ax=axes[2])
    plt.tight_layout(); plt.show()

Блок 2. Обучение линейной модели от одного признака

Признак — median_house_value (сильнейший предиктор Median_Income, r ≈ 0.69). Чтобы стабилизировать обучение в Keras, x масштабируется StandardScaler; веса потом пересчитываются в исходные единицы и выводится формула y = w·x + b.

In [ ]:
x_col = "median_house_value"

X_tr = X_train_s[[x_col]].values.astype("float32")
X_te = X_test_s[[x_col]].values.astype("float32")
y_tr = y_train.values.astype("float32").reshape(-1, 1)
y_te = y_test.values.astype("float32").reshape(-1, 1)

scaler_x = StandardScaler().fit(X_tr)
Xs_tr = scaler_x.transform(X_tr).astype("float32")
Xs_te = scaler_x.transform(X_te).astype("float32")

tf.keras.utils.set_random_seed(42)
model_1f = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(1),                     # y = w_scaled * xs + b
])
model_1f.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
                 loss="mse", metrics=["mae"])
history = model_1f.fit(Xs_tr, y_tr, validation_split=0.2,
                       epochs=200, batch_size=64, verbose=0)

# Пересчёт весов в исходные единицы
w_s = float(model_1f.layers[0].get_weights()[0].squeeze())
b_s = float(model_1f.layers[0].get_weights()[1].squeeze())
mu, sigma = float(scaler_x.mean_[0]), float(scaler_x.scale_[0])
w_raw = w_s / sigma
b_raw = b_s - w_raw * mu

print(f"Модель: {TARGET} = {w_raw:.6e} * {x_col} + {b_raw:.6f}")

y_pred_tr = model_1f.predict(Xs_tr, verbose=0).ravel()
y_pred_te = model_1f.predict(Xs_te, verbose=0).ravel()

Блок 3. Пункт 1 — график разброса + линия регрессии

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_tr.ravel(), y_tr.ravel(), s=8, alpha=0.35,
           color="steelblue", label="train")
ax.scatter(X_te.ravel(), y_te.ravel(), s=8, alpha=0.35,
           color="seagreen", label="test")

xx = np.linspace(X_tr.min(), X_tr.max(), 100)
yy = w_raw * xx + b_raw
ax.plot(xx, yy, color="crimson", lw=2.5,
        label=f"y = {w_raw:.3e}·x + {b_raw:.3f}")

ax.set_xlabel(x_col); ax.set_ylabel(TARGET)
ax.set_title("Линейная регрессия от одного признака (Keras)")
ax.legend(); plt.tight_layout(); plt.show()

print(f"f(x) = {w_raw:.6e} * x + {b_raw:.6f}")
print(f"Интерпретация: +1 к {x_col} → +{w_raw*1000:.4f} к {TARGET} на 1000 единиц x")

Ожидаемо: w_raw ≈ 2·10⁻⁵, b_raw ≈ 1.5–2.0 — с ростом стоимости дома растёт и медианный доход, но разброс очень широкий: визуально точки плотным облаком, линия объясняет лишь часть разброса (R² порядка 0.45–0.50).

Блок 4. Пункты 2–4 — плотность/гистограмма ошибок, pred/true, квадрат ошибки, метрики

In [ ]:
plot_diagnostics(y_tr, y_pred_tr, title="Train · single feature")
plot_diagnostics(y_te, y_pred_te, title="Test · single feature")

met_tr = compute_metrics(y_tr, y_pred_tr)
met_te = compute_metrics(y_te, y_pred_te)

print_metrics("TRAIN · single feature", met_tr)
print_metrics("TEST  · single feature", met_te)

Ожидаемые значения: MAE ≈ 0.55–0.60, RMSE ≈ 0.75–0.80, R²(train) ≈ 0.48–0.50, R²(test) чуть ниже — примерно совпадает с train, переобучения нет (одна линейная связь).

**Комментарий**

LLM по-прежнему дает слабо привязанные к реальности оценки. В сравнение с GPT Luna метрики лучше, однако текущая модель обучалась дольше 200 эпох против 80, но и имеет меньший batch_size 64 против 512

In [ ]:
models_r2_data[1][0] = r2_score(y_te, y_pred_te)

### Построение линейной регрессионной модели от нескольких признаков (переменных) в соответствии с рекомендацией от LLM и выполнение анализа значимости и вклада выбранных признаков.

**ПРОМПТ**:

```
Построй линейную регрессионную модель от нескольких признаков и выполни анализ значимости и вклада выбранных признаков. Модель без скрытых слоев.
Для анализа используй ранее введенные функции.
Выведи формулу y=f(X)
```

* Включен режим DeepThink

**Ответ DeepSeek**

Блок 1. Обучение Keras-модели (один Dense-слой, без скрытых слоёв)

Используем ранее подготовленные выборки X_train_m / X_test_m / y_train / y_test и ранее определённые функции. Признаки масштабируются StandardScaler, веса затем пересчитываются в исходные единицы. Модель — ровно одна Dense(1) без активации (чистая линейная регрессия).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

features = ["median_house_value", "Distance_to_coast",
            "Tot_Rooms", "Median_Age", "ocean_proximity_INLAND"]

X_tr = X_train_m[features].values.astype("float32")
X_te = X_test_m[features].values.astype("float32")
y_tr = y_train.values.astype("float32").reshape(-1, 1)
y_te = y_test.values.astype("float32").reshape(-1, 1)

scaler_X = StandardScaler().fit(X_tr)
Xs_tr = scaler_X.transform(X_tr).astype("float32")
Xs_te = scaler_X.transform(X_te).astype("float32")

tf.keras.utils.set_random_seed(42)
model_multi = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(features),)),
    tf.keras.layers.Dense(1),                          # без скрытых слоёв
])
model_multi.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.03),
                    loss="mse", metrics=["mae"])
history = model_multi.fit(Xs_tr, y_tr, validation_split=0.2,
                          epochs=300, batch_size=64, verbose=0)

Блок 2. Веса в исходных единицах и формула y = f(X)

In [ ]:
w_s   = model_multi.layers[0].get_weights()[0].ravel()
b_s   = float(model_multi.layers[0].get_weights()[1].squeeze())
mu    = scaler_X.mean_.ravel()
sigma = scaler_X.scale_.ravel()

w_raw = w_s / sigma
b_raw = b_s - float(np.sum(w_raw * mu))

coef_table = pd.DataFrame({
    "feature": features,
    "w_raw":   w_raw,
    "w_scaled": w_s,
})
print(coef_table.to_string(index=False, float_format=lambda v: f"{v:.6e}"))
print(f"\nintercept b = {b_raw:.6f}")

# Формула y = f(X)
parts = [f"{b_raw:.4f}"] + [f"({w:+.4e})·{f}" for f, w in zip(features, w_raw)]
print("\nMedian_Income = " + " +\n    ".join(parts))

Ожидаемые веса (порядок величин):

- median_house_value ≈ +2·10⁻⁵ (главный положительный вклад),
- Distance_to_coast ≈ −5·10⁻⁷ (дальше от побережья — ниже доход),
- Tot_Rooms ≈ +2·10⁻⁴ (больше комнат — выше доход),
- Median_Age ≈ −0.01,
- ocean_proximity_INLAND ≈ −0.3…−0.4 (INLAND резко снижает доход).



Блок 3. Диагностика и метрики (переиспользование функций)

In [ ]:
y_pred_tr = model_multi.predict(Xs_tr, verbose=0).ravel()
y_pred_te = model_multi.predict(Xs_te, verbose=0).ravel()

plot_diagnostics(y_tr, y_pred_tr, title="Train · multi feature")
plot_diagnostics(y_te, y_pred_te, title="Test  · multi feature")

print_metrics("TRAIN · multi feature", compute_metrics(y_tr, y_pred_tr))
print_metrics("TEST  · multi feature", compute_metrics(y_te, y_pred_te))

Ожидаемый результат относительно модели от одного признака: MAE ↓ с ~0.55 до ~0.45, RMSE ↓ с ~0.78 до ~0.60, R² ↑ с ~0.48 до ≈ 0.62–0.65. Это подтверждает, что добавленные признаки дают существенный прирост.

In [ ]:
models_r2_data[1][1] = r2_score(y_te, y_pred_te)

Блок 4. Анализ значимости (statsmodels OLS на тех же X, y)

Keras не выдаёт p-значений напрямую. Поскольку Keras с одним Dense(1) и MSE-лоссом минимизирует тот же функционал, что OLS, коэффициенты совпадают с точностью до численной сходимости. Поэтому стандартную таблицу значимости получаем из statsmodels.OLS на тех же признаках.

In [ ]:
X_ols = sm.add_constant(X_tr.astype("float64"))
ols   = sm.OLS(y_tr.ravel().astype("float64"), X_ols).fit()

In [ ]:
# До
# sig_table = pd.DataFrame({
#     "feature":     ["const"] + features,
#     "coef":        ols.params,
#     "std_err":     ols.bse,
#     "t_stat":      ols.tvalues,
#     "p_value":     ols.pvalues,
#     "CI_low(2.5%)":  ols.conf_int()[0],
#     "CI_high(97.5%)": ols.conf_int()[1],
# })
sig_table = ols.summary2().tables[1].reset_index()
print(sig_table.to_string(index=False, float_format=lambda v: f"{v:.4e}"))
print(f"\nR² (OLS) = {ols.rsquared:.4f}   Adj.R² = {ols.rsquared_adj:.4f}")
print(f"F-статистика = {ols.fvalue:.2f},  p = {ols.f_pvalue:.2e}")

Как читать таблицу:
- p_value < 0.05 — признак статистически значим;
- t_stat по модулю — относительная «сила» сигнала с учётом масштаба и разброса;
- CI_low / CI_high — 95 % доверительный интервал коэффициента; если интервал не содержит 0, признак значим.

Ожидаемо: у всех пяти признаков p ≈ 0, т.к. N_train ≈ 16 500 наблюдений — даже слабые эффекты (как у Tot_Rooms) становятся значимыми. Смотрим больше на величину коэффициента и на стандартизованный вклад (блок 5).

Блок 5. Вклад признаков (стандартизованные коэффициенты и VIF)

In [ ]:
# Стандартизованные коэффициенты β = w_i · σ_x_i / σ_y  →  безразмерны, сопоставимы между признаками
std_x = X_tr.std(axis=0, ddof=0)
std_y = y_tr.std(ddof=0)
beta  = w_raw * std_x / std_y

contrib = pd.DataFrame({
    "feature":     features,
    "beta":        beta,
    "|beta|":      np.abs(beta),
    "share_%":     100 * np.abs(beta) / np.abs(beta).sum(),
}).sort_values("|beta|", ascending=False)
print(contrib.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# VIF — проверить, что мультиколлинеарность не искажает веса
X_df = pd.DataFrame(X_tr, columns=features).astype("float64")
vif = pd.DataFrame({
    "feature": features,
    "VIF":     [variance_inflation_factor(X_df.values, i) for i in range(len(features))],
})
print("\nVIF:\n" + vif.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

# ВРУЧНУЮ
contrib_sorted = contrib.sort_values("|beta|", ascending=True)
sign_share = contrib_sorted["share_%"] * np.sign(contrib_sorted["beta"])

# Визуализация вкладов
fig, ax = plt.subplots(figsize=(9, 4))
# order = np.argsort(np.abs(beta))
# ax.barh(np.array(features)[order], beta[order],
        # color=["crimson" if b < 0 else "seagreen" for b in beta[order]])
ax.barh(
    contrib_sorted["feature"], 
    sign_share,
    color=["crimson" if b < 0 else "seagreen" for b in contrib_sorted["beta"]]
)
ax.axvline(0, color="k", lw=1)
# ax.set_xlabel("Standardized coefficient β")
ax.set_xlabel("Вклад признака в % (со знаком)")
ax.set_title("Относительный вклад признаков в Median_Income")
plt.tight_layout(); plt.show()

Интерпретация:
- β показывает, на сколько стандартных отклонений изменится Median_Income при изменении признака на одно его стандартное отклонение — это сопоставимая мера вклада.
- share_% — доля модуля вклада признака в сумме модулей (условная «важность»).
- VIF < 5 — мультиколлинеарность отсутствует (мы специально оставили по одному представителю от коллинеарных групп). Если бы VIF превышал 10 у какого-то признака, это сигнализировало бы, что веса нестабильны и часть признаков нужно убрать.

Блок 6. Итоговый анализ

Формула модели в исходных единицах (см. блок 2) показывает, что наибольший по модулю «сырой» вклад даёт median_house_value, но при переходе к стандартизованным коэффициентам относительный вклад распределяется так (ожидаемо): median_house_value ≈ 55–60 %, ocean_proximity_INLAND ≈ 15–20 %, Distance_to_coast ≈ 10–15 %, Median_Age ≈ 5–7 %, Tot_Rooms ≈ 3–5 %.

Значимость (statsmodels): все пять признаков статистически значимы (p < 0.001). F-тест модели отклоняет гипотезу «все коэффициенты нулевые». R² модели ≈ 0.63 — рост относительно R²_single ≈ 0.48, то есть добавленные признаки объясняют дополнительно ≈ 15 п.п. дисперсии Median_Income.

Ограничения:
- Ошибки по-прежнему гетероскедастичны (видно на pred vs true и squared error vs true): дисперсия ошибок растёт с уровнем цели. Это — аргумент в пользу log1p(Median_Income) как цели, либо взвешенного МНК / деревьев.
- Остатки имеют острый пик — целевой признак Median_Income в датасете ограничен сверху (cap 15.0001), что срезает правый хвост. Удаление cap-точек на этапе IQR-фильтра (k=1.5) частично это исправило, но лёгкая асимметрия сохраняется.

**Комментарий**

DeepSeek провел аналогичный Luna анализ, однако вновь предоставил нерабочий код - пытался инициализировать sig_table вручную неправильными по размеру данными, хотя есть специальный метод. График блока 5 вручную был изменен под показ % вклада признаков

### Построение регрессионной модели на основе многослойного персептрона

**ПРОМПТ**:

```
Построй регрессионную модель на основе многослойного персептрона множества признаков. Построй график изменения ошибки от эпох обучения. Проведи аналогичный анализ точности модели: pred и true, квадрат ошибки, метрики. Оцени вклад каждого признака.
```

* Включен режим DeepThink

**Ответ DeepSeek**


Блок 1. Архитектура MLP и обучение

Используем те же выборки и функции из предыдущих блоков. Архитектура: Dense(64) → Dense(32) → Dense(16) → Dense(1) с relu и лёгким Dropout. Для стабильности — EarlyStopping по val_loss и ReduceLROnPlateau.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

features = ["median_house_value", "Distance_to_coast",
            "Tot_Rooms", "Median_Age", "ocean_proximity_INLAND"]

X_tr = X_train_m[features].values.astype("float32")
X_te = X_test_m[features].values.astype("float32")
y_tr = y_train.values.astype("float32").reshape(-1, 1)
y_te = y_test.values.astype("float32").reshape(-1, 1)

scaler_X = StandardScaler().fit(X_tr)
Xs_tr = scaler_X.transform(X_tr).astype("float32")
Xs_te = scaler_X.transform(X_te).astype("float32")

scaler_y = StandardScaler().fit(y_tr)                 # масштабируем и цель — ускоряет сходимость MLP
ys_tr = scaler_y.transform(y_tr).astype("float32")
ys_te = scaler_y.transform(y_te).astype("float32")

tf.keras.utils.set_random_seed(42)

model_mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(features),)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1),
])
model_mlp.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="mse", metrics=["mae"])
model_mlp.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=25,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=10, min_lr=1e-5),
]
history = model_mlp.fit(
    Xs_tr, ys_tr, validation_split=0.2,
    epochs=300, batch_size=64, verbose=0, callbacks=callbacks,
)
print(f"Эпох обучения: {len(history.history['loss'])}")

Блок 2. График изменения ошибки по эпохам

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(history.history["loss"],     label="train MSE", color="steelblue")
axes[0].plot(history.history["val_loss"], label="val MSE",   color="darkorange")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE (scaled y)")
axes[0].set_title("MSE по эпохам"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history["mae"],     label="train MAE", color="steelblue")
axes[1].plot(history.history["val_mae"], label="val MAE",   color="darkorange")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("MAE (scaled y)")
axes[1].set_title("MAE по эпохам"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

Ожидаемо: обе кривые быстро падают первые 30–50 эпох, затем выходят на плато; val_loss держится близко к train_loss — переобучения почти нет (Dropout + EarlyStopping). Если val_loss начинает расти раньше, чем останавливается обучение, EarlyStopping вернёт лучшие веса — это и есть сигнал, что дальше модель запоминает шум.

Блок 3. Предсказания в исходном масштабе цели

In [ ]:
y_pred_mlp_tr = scaler_y.inverse_transform(model_mlp.predict(Xs_tr, verbose=0)).ravel()
y_pred_mlp_te = scaler_y.inverse_transform(model_mlp.predict(Xs_te, verbose=0)).ravel()

Блок 4. Диагностика через ранее определённые функции + метрики

In [ ]:
plot_diagnostics(y_tr, y_pred_mlp_tr, title="Train · MLP")
plot_diagnostics(y_te, y_pred_mlp_te, title="Test  · MLP")

print_metrics("TRAIN · MLP", compute_metrics(y_tr, y_pred_mlp_tr))
print_metrics("TEST  · MLP", compute_metrics(y_te, y_pred_mlp_te))

# Сравнение с линейной multi-моделью
print_metrics("TEST  · linear multi (для сравнения)",
              compute_metrics(y_te, y_pred_te))

In [ ]:
models_r2_data[1][2] = r2_score(y_te, y_pred_mlp_te)

Ожидаемый результат: MLP даёт MAE ≈ 0.40–0.42, RMSE ≈ 0.55–0.58, R²(test) ≈ 0.68–0.72 — на 4–7 п.п. выше линейной multi-модели. График pred vs true заметно плотнее вокруг y = x; squared error vs true по-прежнему растёт с true, но абсолютный уровень ошибок ниже.

Блок 5. Вклад признаков — Permutation Importance

Для MLP веса отдельных нейронов не имеют прямой интерпретации. Корректный способ — permutation importance: по очереди перемешиваем один признак в тесте и смотрим, насколько растёт ошибка. Используем R² как метрику (чем сильнее падение, тем важнее признак).

In [ ]:
def permutation_importance_keras(model, scaler_X, scaler_y, X, y, features,
                                n_repeats=10, random_state=42):
    rng = np.random.default_rng(random_state)
    X = X.copy()
    y = y.ravel()
    base_pred = scaler_y.inverse_transform(model.predict(scaler_X.transform(X), verbose=0)).ravel()
    base_r2 = r2_score(y, base_pred)
    results = []
    for j, f in enumerate(features):
        scores = []
        for _ in range(n_repeats):
            Xp = X.copy()
            rng.shuffle(Xp[:, j])
            pred = scaler_y.inverse_transform(model.predict(scaler_X.transform(Xp), verbose=0)).ravel()
            scores.append(base_r2 - r2_score(y, pred))
        results.append({
            "feature": f,
            "mean_dR2": float(np.mean(scores)),
            "std_dR2":  float(np.std(scores)),
        })
    return pd.DataFrame(results).sort_values("mean_dR2", ascending=False), base_r2

perm_df, base_r2 = permutation_importance_keras(
    model_mlp, scaler_X, scaler_y, X_te, y_te, features, n_repeats=10
)
print(f"Base R² (test): {base_r2:.4f}\n")
print(perm_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Визуализация
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(perm_df["feature"][::-1], perm_df["mean_dR2"][::-1],
        xerr=perm_df["std_dR2"][::-1],
        color="steelblue", ecolor="black", capsize=4)
ax.set_xlabel("ΔR² при перемешивании признака")
ax.set_title("Permutation importance (MLP, test)")
plt.tight_layout(); plt.show()

Ожидаемый порядок важности:
- median_house_value — ΔR² ≈ 0.40–0.45 (доминирует, как и в линейной модели).
- ocean_proximity_INLAND — ΔR² ≈ 0.05–0.10 (категориальный сигнал, который линейная модель тоже ловила).
- Distance_to_coast — ΔR² ≈ 0.03–0.06.
- Median_Age — ΔR² ≈ 0.01–0.03.
- Tot_Rooms — ΔR² ≤ 0.01.

Порядок совпадает с линейной моделью (по β и по share_%), что подтверждает устойчивость выводов. Отличие в том, что MLP распределяет важность между признаками более «размазанно» (нелинейные взаимодействия), но иерархия сохраняется.

Блок 6. Анализ и выводы
- Кривая обучения — без переобучения: val_loss и train_loss идут параллельно; ранняя остановка срабатывает около 100–150 эпох. Dropout + ReduceLROnPlateau ускоряют выход на плато.
- Метрики: MLP > линейной multi-модели по всем метрикам — R² ≈ 0.70 против ≈ 0.63, MAE ≈ 0.41 против ≈ 0.45. Прирост не гигантский, потому что основная информация о доходе сосредоточена в median_house_value, а зависимость доход↔стоимость близка к линейной. MLP выигрывает за счёт улавливания слабых нелинейностей (например, нелинейный эффект Median_Age, взаимодействие Distance_to_coast × ocean_proximity_INLAND).
- Гетероскедастичность сохраняется — на squared error vs true видно, что у «дорогих» районов ошибки больше. Для её устранения — переход к log1p(y) или взвешенному лоссу.
- Permutation importance устойчиво ставит median_house_value на первое место, ocean_proximity_INLAND и Distance_to_coast — на второе-третье; вклад Tot_Rooms мал (< 0.01 ΔR²). Это согласуется с линейной моделью и подтверждает, что 5 признаков достаточны для базовой точности, а для дальнейшего роста R² нужны дополнительные признаки или более сложная инженерия (log1p(median_house_value), отношения, полиномы).

**Комментарий**:

В ЛР3 (на 2025) наилучший полученный перцептрон состоял из 4 скрытых слоев - $1000 \to 500 \to 200 \to 10$ (605,721 параметров) и достигал $R^2=0.534$. LLM же выбрал модель на 3 скрытых слоя с 2 dropout и колбэками на уменьшение `learning_rate` и раннюю остановку (3,009 параметра). Полученная LLM модель хуже по метрике $R^2$, но обучалась столько же эпох.

### Сравнение с ЛР-3

**ПРОМПТ**

```
Я ранее строил MLP-модель с другими признаками:

features = ['Median_House_Value', 'Distance_to_coast', 'Tot_Rooms', 'Median_Age', 'Population']

def initNewModel(size=5):
    global totalHistoryLossTrain
    totalHistoryLossTrain=[]
    global totalHistoryLossTest
    totalHistoryLossTest=[]

    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(shape=(size,)))

    model.add(tf.keras.layers.Dense(units=1000,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=500,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=200,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=10,  activation=tf.keras.activations.relu))

    model.add(tf.keras.layers.Dense(units=1,  activation=None))

    fLoss=tf.keras.losses.MeanSquaredError()
    fOptimizer=tf.keras.optimizers.Adam(learning_rate=0.01)
    fMetricList  = [tf.keras.losses.MeanSquaredError(), tf.keras.losses.MeanAbsoluteError()]
    model.compile(
        loss      =fLoss,
        optimizer =fOptimizer,
        metrics   = fMetricList
    )
    return model


И проводил эксперименты
Эксперименты с batch_size:

Признаки	Структура	ЭпохОбучения	Размер	R2	RMSE
0	[Median_House_Value, Distance_to_coast, Tot_Ro...	5вх-> 4 ->1вых	100	64	0.528724	0.131514
1	[Median_House_Value, Distance_to_coast, Tot_Ro...	5вх-> 4 ->1вых	100	128	0.509933	0.134111
2	[Median_House_Value, Distance_to_coast, Tot_Ro...	5вх-> 4 ->1вых	100	256	0.534347	0.130727
3	[Median_House_Value, Distance_to_coast, Tot_Ro...	5вх-> 4 ->1вых	100	1024	0.527885	0.131631

Эксперименты с epoch:

Признаки	Структура	ЭпохОбучения	R2	RMSE
0	[Median_House_Value, Distance_to_coast, Tot_Ro...	5вх-> 4 ->1вых	20	0.473984	0.138942
1	[Median_House_Value, Distance_to_coast, Tot_Ro...	5вх-> 4 ->1вых	80	0.511734	0.133864
2	[Median_House_Value, Distance_to_coast, Tot_Ro...	5вх-> 4 ->1вых	160	0.531746	0.131092
3	[Median_House_Value, Distance_to_coast, Tot_Ro...	5вх-> 4 ->1вых	500	0.480007	0.138145


Твои реузльтаты по MLP:

[TRAIN · MLP]
  MAE  : 0.784548
  MSE  : 0.994537
  RMSE : 0.997265
  R2   : 0.523403
[TEST  · MLP]
  MAE  : 0.786540
  MSE  : 1.014140
  RMSE : 1.007045
  R2   : 0.522511
[TEST  · linear multi (для сравнения)]
  MAE  : 0.833959
  MSE  : 1.128659
  RMSE : 1.062384
  R2   : 0.468592

Сравни свои результаты с моими, проинтерпритируй разницу структуры MLP / параметров обучения и их влияние на метрики.
```

* DeepThink

**Ответ Deepseek**

1. Ключевой момент: RMSE сравнивать нельзя — только R²

Твои $RMSE ≈ 0.13$, мои $RMSE ≈ 1.01$ — это артефакт разного масштаба целевой переменной, а не разница в качестве. Ты, судя по масштабу, используешь нормализованный y ($MinMax → [0,1]$, либо $y_{std} = 1$). Я тоже обучаю MLP на стандартизованном y, но метрики считаю через scaler_y.inverse_transform(...), т.е. в исходных единицах Median_Income. При $σ(Median_Income) ≈ 1.4$ это ровно и даёт $RMSE ≈ 1.0$.

Проверка через алгебру: $RMSE_{std} ≈ σ_y · √(1 − R²)$.

- У тебя: $R² ≈ 0.53 → RMSE_{std} ≈ 0.685·σ_y$. Чтобы это стало 0.13, σ_y твоего y-нормированного ≈ 0.19 — согласуется с MinMax на скошенном доходе.
- У меня: $R² ≈ 0.52 → RMSE_{raw} ≈ 0.685·1.4 ≈ 0.96$, реально 1.01 — сходится.

Вывод №1: прямое сравнение RMSE не имеет смысла, пока y масштабированы по-разному. Единственная сопоставимая метрика — R² (и, условно, MAE после деления на σ_y). По R² результаты очень близки: твои лучшие 0.528–0.534 против моих 0.523.

2. Сравнение архитектур: 200-кратная разница в параметрах — и ноль прироста R²

| Твоя |	Моя |
| -| -|
|Слои	| 1000 → 500 → 200 → 10 → 1 |	64 → 32 → 16 → 1 |
| Dropout	| нет |	0.10 (первые два)| 
| Параметров (≈)	|608 000	|3 000|
| Параметров на train-строку	|≈ 37|	≈ 0.18|
| lr	|1e-3? нет — 1e-2|	1e-3|
| Регуляризация	|только «своими руками» (эпохи)|	Dropout + EarlyStopping + ReduceLROnPlateau|
| y	|нормированный	|стандартизованный, но метрики в исходных единицах|
| R² (test)	| 0.534 |	0.523|
| ΔR²	| —	| −1.1 п.п. |

Это классическая иллюстрация, что для табличных данных с 5 признаками большие сети бесполезны. 608 тыс. параметров объясняют данные ничуть не лучше 3 тыс. Основной сигнал в задаче почти линейный (Median_House_Value → Median_Income, r≈0.69), и «глубина» его не обогащает.

Почему 1000–500–200–10 без Dropout не развалилось:

- Задача «прибита» к Median_House_Value, сеть быстро выучивает почти линейную зависимость и в этом режиме не переобучается даже при избыточной ёмкости — есть много весов ≈ 0, они просто не участвуют.

- Один только lr=1e-2 на нормированном y (диапазон [0,1]) эффективно соответствует lr≈1e-3 на стандартизованном — это, скорее всего, компенсирует отсутствие BatchNorm и большой шаг обучения.

- Но потенциал переобучения «спрятан» и обнажается в эксперименте с эпохами.

3. Эксперимент с эпохами — прямое подтверждение необходимости EarlyStopping

| Эпох|	R²|
|-|-|
| 20|	0.474|
| 80|	0.512|
| 160|	0.532|
| 500|	0.480 ⬇|

Между 160 и 500 эпохами R² падает на 5 п.п. Это ровно тот режим, от которого защищает EarlyStopping(patience=25) — без него невозможно понять, где остановиться, и модель «уезжает» в запоминание шума. Если бы ты добавил EarlyStopping к своей архитектуре, скорее всего, получил бы $R² ≈ 0.54–0.55$ с тем же качеством, но без риска.

Мой $patience=25$ как раз даёт остановку раньше «дна» валидационной кривой. Мой график «loss по эпохам» с плато и стабильной val-кривой — это визуальный индикатор, что 300 эпох с запасом, а фактическая остановка происходит ≈ 120–150. Эксперимент с 500 эпохами доказывает, что без этого колбэка я бы деградировал.

4. Эксперимент с batch_size — почти плоская поверхность

|batch|	R²|
|-|-|
|64|	0.529|
|128|	0.510 ⬇ (шум)|
|256|	0.534|
|1024|	0.528|

Разброс 0.51–0.53 — в пределах шума train/test-сплита. Для N=16.5K и lr=1e-2 batch_size не является значимым гиперпараметром, потому что шагов градиента на эпоху при любом из этих размеров достаточно для сходимости за 100+ эпох. Мой batch_size=64 — консервативный дефолт, на результат не влияет.

5. Разные признаки: Population vs ocean_proximity_INLAND

|Твои признаки	|Мои признаки|
|-|-|
|Median_House_Value|	median_house_value|
|Distance_to_coast	|Distance_to_coast|
|Tot_Rooms	|Tot_Rooms|
|Median_Age	|Median_Age|
|Population (r = 0.005)|	ocean_proximity_INLAND (r = −0.24)|

Замена признака с почти нулевой корреляцией на признак со средней корреляцией не дала прироста R². Это информативно:

- Population в твоей модели — фактически шум. Сеть с 600К параметров тратит на него ёмкость, но это не улучшает обобщение (и не ухудшает — просто не вклад).

- ocean_proximity_INLAND сильно дублирует Distance_to_coast: «inland» — это по сути «далеко от побережья». Мультиколлинеарности формально нет (|r| < 0.8), но информационного прироста в присутствии Distance_to_coast тоже почти нет. Именно поэтому R² остаётся ≈ 0.52.

Вывод №3: 5 признаков для этой задачи упираются в потолок R²≈0.52–0.54 независимо от архитектуры. Для заметного роста нужны новые признаки (полиномы, log1p(median_house_value), отношения Tot_Rooms/Households), а не более широкая сеть и не замена Population на INLAND.

6. Моё предыдущее предсказание R² ≈ 0.68–0.72 было неверным — почему

В прошлом ответе я оценил ожидаемый R² MLP в 0.68–0.72, реальный тест — 0.523. Признаю ошибку; вот её источники:

- Я недооценил эффект IQR-фильтра по Median_Income (k=1.5): удаление «хвостов» дохода вычищает самые предсказуемые точки (высокий доход ↔ дорогое жильё), снижая верхнюю границу достижимого R².

- Я переоценил вклад ocean_proximity_INLAND: он коррелирует с целью (−0.24), но почти полностью «покрыт» Distance_to_coast, а Distance_to_coast, в свою очередь, не так сильно нагружен сигналом, как я полагал.

- Разброс дохода внутри любого median_house_value огромен (гетероскедастичность): при R²_single ≈ 0.47 уже один признак объясняет почти всю доступную линейную дисперсию. Четыре оставшихся признака добавляют +5.5 п.п. к R², но не +20.

Твой независимый эксперимент с 600К параметрами, 4 разными batch_size и 4 разными epoch — это хорошая эмпирическая проверка потолка, и она подтверждает: потолок ≈ 0.53 для выбранного набора признаков.

7. Сводная интерпретация

|Что меняется|	Влияние на R²|
|-|-|
|Архитектура 1000-500-200-10 → 64-32-16 (200× меньше параметров)|	≈ 0 (−1 п.п.)|
|batch_size 64 ↔ 1024	|≈ 0 (шум ±1 п.п.)|
|epochs 160 → 500|	−5 п.п. (переобучение)|
|Population → INLAND|	≈ 0 (информационная избыточность)|
|lr 1e-3 (меня) ↔ 1e-2 (твои, на нормированном y)|	≈ 0 (эффективная динамика схожа)|
|EarlyStopping / Dropout vs. «без ничего»	|защита от −5 п.п., не прирост|

Итоговая мысль: на этой задаче качество определяется инженерией признаков, а не ёмкостью MLP. Твой эксперимент — убедительное доказательство, что большая сеть не нужна. Мой MLP (3К параметров) даёт то же R², что твой (608К), и это ожидаемо: при 16.5 тыс. строк, 5 признаках и почти линейной основной зависимости добавление параметров бесполезно. Следующий шаг для роста R² — не архитектура, а новые признаки (полиномы, отношения, log1p целевой), либо переход к градиентному бустингу (XGBoost/LightGBM/CatBoost) — на таких данных он типично даёт R² на 5–10 п.п. выше любой MLP.

**Комментарий**

DeepSeek как и GPT указал, что сложная структура может объяснять сложные зависимости лучше, однако склонна к переобучению. В данном же случае разница в $R^2$ настолько мала, что сложная структура попросту избыточна. DeepSeek заметил, что ранее включал к независимым признакам категориальный ocean_proximity_inland, который по смыслу близок к distance_from_ocean - шум, нужна замена

### Ансамблевые модели

**ПРОМПТ**:
```
Для сравнения построй ансамблевую регрессионную модель. Обоснуй выбор типа ансамбля и его структуры.
```

* DeepThink

1. Обоснование выбора типа ансамбля

Кандидаты для табличной регрессии на 5 признаках, N≈16.5K:

|Тип|	Представитель|	Плюсы|	Минусы на этой задаче|
|-|-|-|-|
|Bagging|	Random Forest|	Устойчив, почти без тюнинга	|Хуже ловит слабые сигналы; усреднение не «исправляет» ошибки последовательно|
|Boosting|	LightGBM / XGBoost / CatBoost	|SOTA на табличных, ловит нелинейности и взаимодействия, быстрый	|Требует контроля переобучения (early stopping, ограничение глубины)|
|Stacking|	LR + RF + LGBM → meta-LR|	Максимальная точность|	Сложно интерпретировать, избыточно при 5 признаках и N=16.5K|

Выбираю Gradient Boosting (LightGBM). Обоснование:

- Задача табличная, регрессия, гетероскедастичная — это классический сценарий для boosting. Он последовательно исправляет остатки предыдущих деревьев, что даёт выигрыш по сравнению с параллельным bagging, когда признаки слабые (как у нас — R² базовой линейной модели всего 0.47).
- Не нужен масштабированиe признаков — важно для сравнения: предсказание не зависит от StandardScaler, что исключает ещё один источник расхождения с линейной моделью.
- Автоматически строит взаимодействия (например, Distance_to_coast × ocean_proximity_INLAND — избыточный сигнал, о котором я писал ранее; boosting сам решит, какой из признаков в узле информативнее).
- LightGBM vs XGBoost vs CatBoost: категориальный признак только один (one-hot INLAND), поэтому преимущество CatBoost в категориальных фичах не работает. LightGBM быстрее и легче, с хорошими дефолтами.
- Stacking отклоняю — при N=16.5K и 5 признаках meta-модель склонна выучить шум базовых моделей. Стек логичнее добавить как следующий шаг, если GB покажет, что базовые модели ошибаются по-разному (проверю через корреляцию ошибок).

2. Обоснование структуры и гиперпараметров

|Параметр	|Значение	|Почему|
|-|-|-|
|n_estimators|	2000 + early_stopping(50)|	Заведомо много; останавливаемся по val_rmse|
|learning_rate|	0.05|	Стандарт; при 0.01 нужно больше деревьев, при 0.1 — риск переобучения|
|num_leaves|	31|	Умеренная сложность; при 5 признаках листьев больше, чем сигнала, не нужно|
|max_depth|	6	|Ограничение глубины = защита от запоминания|
|min_child_samples|	20	|≈ 0.12% от N — отсекает листья с единичными объектами|
|subsample|	0.8	|Стохастичность по строкам (снижает дисперсию)|
|colsample_bytree|	0.8|	Стохастичность по признакам|
|reg_alpha / reg_lambda|	0.1 / 1.0	|L1/L2-регуляризация весов листьев|
|random_state	|42|	Воспроизводимость — как во всех предыдущих моделях|

In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

features = ["median_house_value", "Distance_to_coast",
            "Tot_Rooms", "Median_Age", "ocean_proximity_INLAND"]

X_tr = X_train_m[features]; X_te = X_test_m[features]
y_tr = y_train.values.ravel();  y_te = y_test.values.ravel()

lgb_model = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.05,
    num_leaves=31, max_depth=6, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_te, y_te)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(0),
    ],
)
print(f"Деревьев после early stopping: {lgb_model.best_iteration_}")

# Кривая обучения (валидационный RMSE по итерациям)
ev = lgb_model.evals_result_["valid_0"]["rmse"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ev, color="darkorange", label="val RMSE")
ax.axvline(lgb_model.best_iteration_, color="k", ls="--",
           label=f"best_iter = {lgb_model.best_iteration_}")
ax.set_xlabel("boosting iteration"); ax.set_ylabel("RMSE (Median_Income)")
ax.set_title("LightGBM: learning curve")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

y_pred_lgbm_tr = lgb_model.predict(X_tr)
y_pred_lgbm_te = lgb_model.predict(X_te)

In [ ]:
plot_diagnostics(y_tr, y_pred_lgbm_tr, title="Train · LightGBM")
plot_diagnostics(y_te, y_pred_lgbm_te, title="Test  · LightGBM")

print_metrics("TRAIN · LightGBM", compute_metrics(y_tr, y_pred_lgbm_tr))
print_metrics("TEST  · LightGBM", compute_metrics(y_te, y_pred_lgbm_te))

# Сводка для сравнения
summary = pd.DataFrame([
    {"model": "linear multi",  **compute_metrics(y_te, y_pred_te)},
    {"model": "MLP (64-32-16)", **compute_metrics(y_te, y_pred_mlp_te)},
    {"model": "LightGBM",      **compute_metrics(y_te, y_pred_lgbm_te)},
])
print(summary.round(4).to_string(index=False))

Ожидаемые значения на этом датасете (с учётом потолка ≈ 0.53, который мы выявили ранее):

|Модель	|R² (test)|	MAE|	RMSE|
|-|-|-|-|
|Linear multi|	≈ 0.47|	≈ 0.83|	≈ 1.06|
|MLP	|≈ 0.52	|≈ 0.79	|≈ 1.01|
|LightGBM|	≈ 0.55–0.58|	≈ 0.74–0.76	|≈ 0.97–0.99|

In [ ]:
# Встроенная важность (gain = суммарный прирост качества от разбиений)
imp_gain = pd.DataFrame({
    "feature": features,
    "importance_gain": lgb_model.booster_.feature_importance(importance_type="gain"),
    "importance_split": lgb_model.booster_.feature_importance(importance_type="split"),
}).sort_values("importance_gain", ascending=False)
imp_gain["share_%"] = 100 * imp_gain["importance_gain"] / imp_gain["importance_gain"].sum()
print(imp_gain.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(imp_gain["feature"][::-1], imp_gain["share_%"][::-1], color="seagreen")
ax.set_xlabel("доля вклада, % (по gain)")
ax.set_title("LightGBM: важность признаков")
plt.tight_layout(); plt.show()

# SHAP — согласованная важность для дерева
import shap

explainer = shap.TreeExplainer(lgb_model)
shap_vals = explainer.shap_values(X_te)
shap.summary_plot(shap_vals, X_te, feature_names=features, plot_type="bar", show=True)
shap.summary_plot(shap_vals, X_te, feature_names=features, show=True)

6. Сравнение с MLP и Linear

Интерпретация ожидаемых результатов:

- LightGBM > MLP ≈ Linear multi > Linear single. Прирост MLP→LightGBM ожидаю небольшой (+2–5 п.п. R²), потому что основная зависимость median_house_value → Median_Income почти линейна. GB выиграет за счёт:

    - нелинейного эффекта Median_Age (квадратичный «провал» дохода в среднем возрасте домов);

    - нелинейного эффекта Distance_to_coast — доход резко падает в первые 5 км от берега, дальше почти не меняется; линейная модель этот излом сглаживает, дерево — сохраняет.

    - взаимодействий ocean_proximity_INLAND × Distance_to_coast на уровне отдельных узлов.

- Потолок R² остаётся тем же (~0.55–0.58), и это подтверждает вывод из твоего эксперимента с 200× большей сетью: при 5 признаках и сигнале, почти полностью сосредоточенном в median_house_value, выигрыш от «более умной» модели ограничен. Дальнейший рост требует новых признаков, а не новых моделей.
- Важность признаков. Ожидаю median_house_value → 70–80 % gain, ocean_proximity_INLAND и Distance_to_coast → по 5–10 %, Median_Age → 3–5 %, Tot_Rooms → <3 %. Это совпадёт с permutation importance MLP и с β линейной модели — устойчивая иерархия, что подтверждает корректность выводов.
- Корреляция ошибок MLP ↔ LightGBM. Если corr(err_mlp, err_lgbm) < 0.8, это аргумент за стеккинг (LR + MLP + LightGBM → meta-модель) — но прирост там ожидаю ещё меньше (+1 п.п.), поскольку обе модели ошибаются на одних и тех же неинформативных точках (дорогие районы с широким разбросом дохода).
- Гетероскедастичность по-прежнему видна на squared error vs true: даже LightGBM не устраняет рост дисперсии ошибок с уровнем true. Это ограничение данных, а не модели.

Итог: LightGBM даст лучший из четырёх результат по R² (≈0.55–0.58), но прирост над MLP скромный и ожидаемо упирается в информационный потолок 5 признаков. Дальнейшее улучшение — инженерия признаков (log1p(median_house_value), отношения Tot_Rooms/Households, полиномы координат) и stacking MLP + LightGBM, а не наращивание ансамбля как такового.

**Комментарий**

DeepSeek в отличие от GPT указал разницу в применимости разных типов ансамблей и явно указал, почему выбрал lightgbm. $R^2$ находится в ожидаемом диапазоне значений. Вывод тот же - влияние слабое, нужно изменять признаки

### Сравнение со ручными ансамблями

**ПРОМПТ**
```
Я строил ансамбли ранее. Привел свои лучшие характеристики.

−Бэггинг  𝐵𝐵 = 1000:
o MAE: с 0.757
o MSE: 0.970
o RMSE: 0.985
o R2: 0.543
o Var: 0.397 (Дисперсия абсолютной ошибки)
- Случайный и лес с максимальной глубиной = 15, долей 𝑚𝑚/𝑝𝑝 = 0.4 и B: = 500
o MAE: 0.745
o MSE: 0.934
o RMSE: 0.966
o R2: 0.560
o Var: 0.378 (Дисперсия абсолютной ошибки)
- Градиентный бустинг с λ = 0.1 и 𝐵𝐵 = 500
o MAE: 0.749
o MSE: 0.956
o RMSE: 0.978
o R2: 0.549
o Var: 0.394 (Дисперсия абсолютной ошибки)

Вот твой выход
[TRAIN · LightGBM]
  MAE  : 0.730449
  MSE  : 0.858440
  RMSE : 0.926520
  R2   : 0.588623
[TEST  · LightGBM]
  MAE  : 0.760643
  MSE  : 0.954680
  RMSE : 0.977077
  R2   : 0.550507
         model    MAE    MSE   RMSE     R2
  linear multi 0.8340 1.1287 1.0624 0.4686
MLP (64-32-16) 0.7865 1.0141 1.0070 0.5225
      LightGBM 0.7606 0.9547 0.9771 0.5505

Проинтерпретируй результаты.
```

*DeepThink

**Ответ Deepseek**

1. Сопоставимость результатов

Теперь сравнение корректное: и у тебя, и у меня MAE/RMSE в исходных единицах Median_Income (MAE ≈ 0.74–0.76, RMSE ≈ 0.97–0.99 — один масштаб). Поэтому метрики можно сравнивать напрямую.

2. Сводная таблица (test)

|Модель|	MAE|	RMSE|	R²|	Var(abs err)|
|-|-|-|-|-|
|Bagging (B=1000)	|0.757|	0.985|	0.543|	0.397|
|Random Forest (depth=15, m/p=0.4, B=500)|	0.745|	0.966|	0.560|	0.378|
|GBM (λ=0.1, B=500)	|0.749	|0.978|	0.549	|0.394|
|LightGBM (мой)	|0.761	|0.977|	0.551|	≈0.39|
|MLP (64-32-16)	|0.787	|1.007|	0.523|	—|
|Linear multi	|0.834	|1.062|	0.469|	—|

3. Ключевая интерпретация: RF побеждает мой LightGBM

Твой Random Forest (R² = 0.560) — лучший результат среди всех, включая мой LightGBM (0.551). Разница 0.9 п.п. — небольшая, но систематическая, и она информативна:

- RF-параметры у тебя хорошо подобраны: depth=15 + m/p=0.4 + B=500 — это классический «глубокое дерево + сильная рандомизация по признакам» рецепт. При 5 признаках m/p=0.4 = 2 признака в узле — очень сильная декорреляция деревьев, что даёт максимальный выигрыш от bagging.

- Boosting переобучается сильнее на слабом сигнале. Мой LightGBM показал train R² = 0.589 против test = 0.551 — разрыв 3.8 п.п. Это классический признак переобучения. У RF разрыв должен быть меньше (bagging + subsample по признакам гасят дисперсию), поэтому RF обобщает лучше, когда сигнал почти линейный и признаков мало.

- early_stopping(50) у меня — слишком мягкий. С lr=0.05 и 2000 деревьями модель успела «выучить» часть шума в train. Если поставить patience=20 или lr=0.03, gap сократился бы, и test R² поднялся бы до ≈0.56 — то есть до уровня RF.

- GBM (λ=0.1, B=500) ≈ LightGBM (0.549 ≈ 0.551) — это валидация: разные реализации бустинга на тех же признаках дают одинаковый результат в пределах шума. Значит, потолок boosting на этом наборе — именно ~0.55, и мой LightGBM его достиг.

4. Bagging vs RF — разница в 1.7 п.п. R², и почему

Bagging (0.543) хуже RF (0.560) — и это не шум, а следствие структуры:

- Bagging усредняет деревья, но строит их на всех признаках (m/p = 1.0). Деревья сильно коррелированы (все «видят» median_house_value первым), усреднение почти не снижает дисперсию.

- RF при m/p = 0.4 заставляет деревья смотреть на разные подмножества признаков, ошибки декоррелируются, дисперсия ансамбля падает на ~30–40 % — отсюда Var(abs err) 0.378 против 0.397.

Это ровно тот эффект, ради которого Random Forest и придуман, и на твоих данных он полностью себя оправдал.

5. Дисперсия абсолютной ошибки — самый показательный столбец

Var(abs err) у тебя: 0.397 (Bagging) → 0.378 (RF) → 0.394 (GBM). RF даёт самую стабильную ошибку — не только меньшую MAE, но и меньший разброс. Это важно для практики: если модель используется для оценки дохода, RF предсказуемее, её ошибки реже «выстреливают» в 2–3 раза выше среднего.

У моего LightGBM Var(abs err) не измерена, но по MAE (0.761) и RMSE (0.977) можно оценить как ≈ 0.39 — как у твоего GBM, что логично: boosting-ансамбли ведут себя одинаково.

6. Что это говорит о потолке

Четыре независимых эксперимента (Bagging, RF, GBM, LightGBM) дают R² в узком коридоре 0.543–0.560. Это сильное свидетельство, что потолок качества на текущих 5 признаках ≈ 0.56, и он достигнут твоим RF. Разница между моим LightGBM (0.551) и твоим RF (0.560) — 0.9 п.п., то есть меньше, чем разброс между двумя фолдами CV. Дальнейшее «наращивание» ансамбля (stacking, ещё больше деревьев, тюнинг lr) даст <1 п.п. — эффект в пределах шума.

7. Где реально прячется прирост

Все ансамбли упираются в одно и то же ограничение: сигнал почти линейный и почти весь в median_house_value. Чтобы поднять R² на 5–10 п.п., нужны:

- Инженерия признаков: log1p(median_house_value), Tot_Rooms / Households, Population / Households, полиномы координат, Distance_to_coast с бинами (первые 5 км, 5–20 км, >20 км) — изломы, которые бустинг ловит, но которые можно «подсказать» явно.

- Stacking RF + LightGBM + MLP — если корреляция их ошибок < 0.8. Ожидаю +1 п.п., не больше.

- Переход к log1p(Median_Income) как цели — если гетероскедастичность критична для бизнес-задачи (не для сравнения метрик).

Итог

- Твой Random Forest — победитель по test R² (0.560) и по стабильности ошибки (Var = 0.378). Это не случайность: параметры depth=15, m/p=0.4 хорошо подобраны, а RF на слабом сигнале обобщает лучше, чем boosting.

- Мой LightGBM (0.551) эквивалентен твоему GBM (0.549) — обе реализации boosting упираются в один потолок. Разрыв train/test 3.8 п.п. у меня — сигнал, что patience стоило сделать меньше, но даже с идеальным early stopping LightGBM не обгонит RF на этих данных.

- Все четыре ансамбля укладываются в R² ∈ [0.543, 0.560] — потолок ≈ 0.56 на текущих признаках подтверждён четырежды.

- Следующий шаг — не новый ансамбль, а новые признаки. Именно они, а не архитектура, двигают R² на этой задаче.

**Комментарий**

Вывод прежний - упираемся в потолок по качеству модели.

## Сводка результатов

Для всех моделей сводилась метрика $R^2$. Под ручными понимаются модели ЛР1/ЛР3 - выбирались наилучшие по метрикам модели

In [ ]:
handmade_models_r2_data = [0.377, 0.465, 0.534]
models_r2_data[2] = handmade_models_r2_data

columns = ['1 признак', 'несколько признаков', 'MLP']
rows = ['GPT Luna', 'Deepseek', 'Ручные']

df_results = pd.DataFrame(models_r2_data, index=rows, columns=columns)
df_results.style.format("{:.3f}")

## Об эффективности использования LLM

В качестве критериев сравнения можно обозначить следующие:
- необходимая вовлеченность пользователя
- обоснованность принятых решений
- доля ошибок/галлюцинаций
- чистота решения
- время на проведение работы/скорость ответа

Количество промптов не учитывалось - оно зависит от вовлеченности пользователя и доли ошибок. Точность моделей тоже - LLM никак не рационализировала выбор архитектуры и предлагала *что-то*: в случае GPT Luna архитектура MLP проще, чем у Deepseek, отсюда и худшие метрики 

Сравнение по всем критериям между GPT Luna и DeepSeek-V4.1-Flash-1017:

#### **необходимая вовлеченность пользователя**

GPT Luna позволяет запускать код непосредственно в чате модели, это же делает сама модель по ходу выполнения работы, получая тем самым реальные данные, имея возможность формировать выходные csv-файлы, изображения без непосредственного вмешательства пользователя за пределами начального промпта.

#### **обоснованность принятых решений**

Имея реальные данные GPT Luna в действительности может принимать решения о дальнейшем ходе работы. Модель DeepSeek в свою очередь предположительно галлюцинирует свои предположения (датасет публичный, скорее всего его анализ есть в сети).

Странно обосновано решение DeepSeek включить категориальный признак INLAND, в ответе до обнаружения ошибки включены были и вовсе 3 категориальных признака из 5, наравне с distance_to_coast.

Самостоятельно пользователь может провести множество эскпериментов для разных архитектур моделей и их разных параметров, подобрав лучшие входные. Однако, скорее всего это так же легко можно возложить на LLM.

#### **доля ошибок/галлюцинаций**

Опять модель Deepseek проигрывает, теряя пустые строки датасета, предлагая неработоспособные фрагменты кода (иногда в силу изменений в API) или путая целевой признак, хотя тот явно указан в приложенном пояснении к датасету. Как выше упоминалось, Deepseek предположительно галлюцинирует свои предположения о данных.

Что одна, что вторая модель часто делали то, чего не просилось в промпте, например, сохраняли датасеты, собранные метрики и изображения.

#### **чистота решения**

Стоит отметить, что Deepseek не пытался переопределять уже созданных функций для анализа. GPT Luna добавила к ним ранее не указанный и непрошеный параметр - путь сохранения диаграммы.

Deepseek однако называет переменные в разных стилях, часто перезаписывая значения уже определенных переменных, импортировал одни и те же библиотеки в пределах одного промпта.

GPT Luna в силу независимости от пользовательского окружения может отдавать небольшие наиболее важные отрывки кода, таблицы, диаграммы (инфографику), оставляя громоздкий код *спрятанным* от пользователя в отдельном ipynb-файле. Однако это не один ipynb-файл на весь диалог, а несколько, каждый на свой промпт, это делает решение *"модульным"*, но часто избыточным.

#### **время на проведение работы/скорость ответа**

Отдать приоритет какой-то конкретной модели сложно: GPT Luna запускает ipynb локально, затрачивая больше времени в силу тренировки моделей (до 7 минут), но получая более качественный ответ, привязанный к реальности.

Deepseek дает ответ на промпт в течении 15-20 секунд, но требует исправлений, более частых проверок ответов и повторных уточняющий промптов - в итоге получается столько же времени.

#### **Итог**

Учитывая вышеописанное, большей практической полезностью и удобством для пользователя обладает модель GPT Luna, которая действительно может вести анализ предложенных данных, совершает меньше ошибок и требуя меньше вовлеченности от пользователя. Вместе с тем, модель затрачивает больше времени на формирование ответа (пользователь однако не обязан бездействовать в это время и может делать что-то еще).

В силу обозначенных проблем (модели делают, что не просят) имеет смысл рассмотреть подход небольших и точных промптов, что особенно удобно, когда не нужно запускать код самостоятельно, а это делает модель сама (например, в окружении GPT Luna), однако на это уйдет больше времени пользователя.